<a href="https://colab.research.google.com/github/carlocamilloni/Structural-Bioinformatics/blob/main/Notebooks/report_7_QM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# REPORT: Quantum chemistry with ORCA

First save a copy of this notebook in your Google Drive (`File -> Save a copy in Drive`).

This is a group report. As in the training notebook (`t07_QM`), the two groups work on different topics:

*   **GROUP 1: the forces that hold biomolecules together.** How strong are a hydrogen bond between two DNA bases, the stacking of two bases on top of each other, or the contact between two aromatic side chains in a protein? Where does this energy come from, and how well do simple models describe it?
*   **GROUP 2: the colour of biomolecules.** Why is the green fluorescent protein green, why is rhodopsin (the protein of vision) purple, and why are flavoproteins yellow? We compute the light absorbed by their chromophores, and how it depends on their chemistry and environment.

The notebook has two parts for each group:

1.  **Running the calculations**: the systems to study are listed in the assignment below; divide them among yourselves as you like, and each runs the cells of Part 1 with his or her own choice.
2.  **The report**: the analysis of the results of the whole group, with your text. The code cells collect the results of all the members and make the figures; write your answers in the text cells, in your own words.

**You will present your work to the class** (about 10 minutes per group, followed by questions; the other group has not studied your topic). For this reason the report asks you, at each step, to make figures that explain what you did. The figures made with matplotlib are saved in the folder `figures` of the group; for the 3D views (molecules, orbitals, proteins) rotate the molecule to a clear orientation and take a screenshot, or open the files in VMD or ChimeraX.

The cells that start with `%%group 1` or `%%group 2` run only for the group you choose below, so you can also use *Runtime -> Run all*.

**Group:** 1 / 2

**Members:**

In [ ]:
from datetime import date

today = date.today()
print(today)

## The assignment

### GROUP 1: the forces that hold biomolecules together

The double helix of DNA, the folded structure of a protein, the binding of a drug to its target: all are held together by **noncovalent interactions**, much weaker than covalent bonds (≈ 100 kcal/mol) but numerous. Each is worth a few kcal/mol, only a few times the thermal energy $k_BT$ = 0.6 kcal/mol: strong enough to give stable structures, weak enough to be broken and formed again during function. In a force field they are described by two simple terms, point charges (Coulomb) and Lennard-Jones; with quantum mechanics we can compute them from the electrons and understand where they come from:

| ingredient | what it is | in a force field |
|:--|:--|:--|
| **electrostatics** | attraction between the positive and negative parts of the two molecules (e.g. the partially positive H of an N–H and the lone pair of an O) | point charges |
| **polarisation** | each molecule deforms the electron cloud of the other | missing (or averaged in the charges) |
| **dispersion** (London forces) | the fluctuating electron clouds of the two molecules attract each other; always attractive, important for large flat molecules | the attractive $-1/r^6$ part of Lennard-Jones |
| **Pauli repulsion** | electron clouds cannot overlap | the repulsive $1/r^{12}$ part of Lennard-Jones |

The goal is to compare the **types of interaction** found in biomolecules: hydrogen bonds, stacking of aromatic rings, interactions between charged groups. We work in two steps.

**Step 1: choose the methods (the whole group, about 5 minutes).** A good method is accurate enough and fast enough. We test candidate methods on two complexes for which a very accurate reference interaction energy is known (CCSD(T) extrapolated to an infinite basis set, from the S22 database: Jurečka et al., *Phys. Chem. Chem. Phys.* 8, 1985 (2006); Takatani et al., *J. Chem. Phys.* 132, 144104 (2010)): the uracil dimer, hydrogen-bonded and stacked. The candidates are the semiempirical method **GFN2-xTB** (seconds, used for the dynamics of the training) and the DFT method **r$^2$SCAN-3c** (about a minute). We also look at what happens if r$^2$SCAN-3c loses its dispersion correction. From then on we use a **fast** method (for scans with many points) and an **accurate** one (for the final numbers).

**Step 2: a catalogue of biological interactions.** Four families of interactions, each with two complexes that contrast with each other:

| family | complex 1 | complex 2 | what the pair shows |
|:--|:--|:--|:--|
| nucleic acids | adenine–thymine Watson–Crick pair (`at_wc`) | guanine–cytosine Watson–Crick pair (`gc_wc`) | two vs three hydrogen bonds: why is GC-rich DNA more stable? |
| aromatic contacts | indole on benzene, stacked (`indole_benzene`, Trp–Phe) | dimethyl sulfide on benzene (`dms_benzene`, Met–Phe) | π–π stacking and S–π contacts in the core of proteins |
| charged groups | acetate–methylguanidinium (`acetate_methylguanidinium`, Asp–Arg salt bridge) | methylammonium on 3-methylindole (`methylammonium_indole`, Lys–Trp cation–π) | which survives in water? |
| protein–DNA recognition | dimethyl phosphate–methylguanidinium (`phosphate_methylguanidinium`, Arg–phosphate) | dimethyl phosphate–methylammonium (`phosphate_methylammonium`, Lys–phosphate) | how do proteins grab the backbone of DNA? Why do they use arginine so often? |

**Planning the work.** Divide the families among yourselves, depending on the size of the group: try to cover all four, and keep the two complexes of a family together, since they are meant to be compared. Each person works in his or her own Colab session, so the calculations of different people run at the same time. Approximate computer times (Colab can be up to two times slower):

| work unit | what is computed | time |
|:--|:--|:--|
| Step 1 (one person, first) | the two uracil complexes with both methods | ≈ 10 min |
| nucleic acids | `at_wc`, `gc_wc` | ≈ 15 min |
| aromatic contacts | `indole_benzene`, `dms_benzene` | ≈ 8 min |
| charged groups | `acetate_methylguanidinium`, `methylammonium_indole` | ≈ 6 min |
| protein–DNA recognition | `phosphate_methylguanidinium`, `phosphate_methylammonium` | ≈ 6 min |
| going further 6.1 | chains of *N*-methylacetamide | ≈ 5 min |
| going further 6.2 | 3 ps of dynamics of one complex | ≈ 10 min per complex |

For example, with **2 people**: Step 1 + nucleic acids + charged groups, and aromatic contacts + protein–DNA recognition + the going further. With **3 or 4 people**: one family each, and Step 1 and the going further to whoever has the shorter families. With **more than 4**: the extra people take Step 1 and the two extensions of section 6. The analysis (Part 2) only reads the results, it takes a few seconds: everybody can run it at the end.

The structures are in `Data/qm/interactions`: the A–T pair and the indole–benzene complex come from S22 (with their reference energies), the others were built and optimised with r$^2$SCAN-3c (the three ion pairs in water, otherwise the proton would jump from the cation to the anion). Dimethyl phosphate is the standard model of the backbone of DNA and RNA: one phosphate with its negative charge, and the two sugars reduced to methyl groups.

**For each complex** (Part 1 below): the interaction energy with the accurate method, in vacuum and in water (as a continuum), split into dispersion and the rest; the energy as a function of the distance between the two molecules with the fast method, in vacuum and in water; the electrostatic potential of the two molecules. Then, all together, the **interaction map** of the group, and a look at where these interactions occur in real proteins and nucleic acids.

### GROUP 2: the colour of biomolecules

Proteins and nucleic acids are colourless: their electrons need ultraviolet light to be excited (training, Part 6). Nature makes colour with **chromophores**, molecules with chains of alternating single and double bonds (*conjugated* systems), whose excitation energy falls in the visible. Often the chromophore is bound inside a protein, which tunes its colour: the same retinal molecule absorbs blue, green or red light in the different opsins of our cones, and the green fluorescent protein (GFP) has been engineered into blue, cyan, yellow and red variants by changing a few residues around its chromophore.

We compute the **excitation energies** (or wavelengths, $\lambda$ = 1239.84 / E(eV)) and **oscillator strengths** $f$ with TD-DFT, always with the same setup (CAM-B3LYP/def2-SVP; structures optimised with GFN2-xTB). Like every approximate method, TD-DFT has errors: for these molecules the excitation energies are too high (the wavelengths too short) by 0.2–0.5 eV, and up to almost 1 eV for anions such as the phenolate of GFP. The *changes* of the excitation energy (with the structure, the environment, the protein) are more reliable than the absolute values, and they are what we study. The colour of a chromophore depends on:

1.  **its chemistry**: the length of the conjugated system, protonation (training, Part 6);
2.  **its geometry**: a conjugated system works best when it is planar; proteins can twist it;
3.  **its environment**: solvent, charged or polar groups nearby;
4.  **its motion**: at room temperature the molecule vibrates, and each structure absorbs at a slightly different energy, which gives the width of the absorption band.

**Six molecules** (files in `Data/qm/chromophores`), in three pairs, each studied with the same four lenses:

| theme | molecules | the biology |
|:--|:--|:--|
| GFP | `hbdi_neutral`, `hbdi_anion`: the chromophore of GFP, neutral (phenol) and anionic (phenolate) | GFP absorbs at 395 nm (neutral form) and 475 nm (anionic form); the S65T mutant, at 489 nm, is mostly anionic |
| light sensors | `psb_5`: a model of retinal (protonated Schiff base with 5 conjugated double bonds); `pyp_thioester`: the chromophore of the photoactive yellow protein (a *p*-coumaric thioester, phenolate) | vision (rhodopsin, 498 nm) and the blue-light sensor of bacteria (photoactive yellow protein, 446 nm): both work by twisting a double bond after absorbing light |
| flavins and the UV | `lumiflavin_ox`: the oxidised flavin; `3_methylindole`: the side chain of tryptophan | flavoproteins are yellow (≈ 445 nm); proteins absorb at 280 nm because of Trp and Tyr |

**And then GFP in its protein**: the chromophore of GFP (PDB 1EMA, the S65T mutant) computed in the electric field of the whole protein, every protein atom and crystal water represented by its point charge from the AMBER force field. By switching off the charges of one residue at a time (a *virtual mutation*) we can see which residues tune the colour of GFP. 

**Planning the work.** This is the longer report: all the calculations together take about 7 hours of computer time, so they must be divided. Each person works in his or her own Colab session, so the calculations of different people run at the same time; and each lens is a separate cell, so the work of one person can be spread over two or three sessions (the calculations already finished are not repeated). Approximate computer times (Colab can be up to two times slower):

| work unit | what is computed | time |
|:--|:--|:--|
| a small molecule (`psb_5`, `pyp_thioester`, `3_methylindole`) | lenses 1–3 (7 TD-DFT calculations) | ≈ 15–20 min |
|  | lens 4, motion (dynamics + 6 TD-DFT) | ≈ 15–20 min |
| a large molecule (`hbdi_neutral`, `hbdi_anion`, `lumiflavin_ox`) | lenses 1–3 | ≈ 30–40 min |
|  | lens 4, motion | ≈ 30 min |
| GFP in its protein | the two forms, in vacuum and in the protein (4 calculations) | ≈ 20 min |
|  | each virtual mutation (both forms) | ≈ 8 min (≈ 75 min for all 9) |
| going further 6.1 | a GFP variant | ≈ 5 min |
| going further 6.2 | the Stokes shift of 3-methylindole | ≈ 40 min |

For example, with **2 people**: one large and two small molecules each, and the GFP calculations split between them (the two forms for one, with half of the mutations; the other half for the other, with `RUN_FORMS = False` and `MY_MUTATIONS`): ≈ 3 h each, over two or three sessions. With **3 people**: one pair of molecules each (the GFP pair ≈ 2 h, the light sensors ≈ 1 h, flavin and indole ≈ 1.5 h); the GFP calculations go to the two people with the shorter pairs, one computing the two forms and the other the mutations (`RUN_FORMS = False`). With **4 people**: the three pairs, and the fourth person runs all of GFP and the going further. With **5 or 6**: one or two molecules each, GFP and its mutations shared among the people with small molecules. Only one person at a time should run a given molecule or a given mutation (they write the same files). The analysis (Part 2) only reads the results, it takes a few seconds: everybody can run it at the end, after `sync_from_drive()`.

### Organising the work

To share the results, one member creates a folder `Task7_group1` (or `Task7_group2`) in Google Drive and shares it with the rest of the group, with editing rights. Every member adds a shortcut to it in his or her own `MyDrive` (right click on the folder -> `Organise` -> `Add shortcut`). The calculations run on the local disk of Colab and their results are copied automatically to this folder; each molecule has its own sub-folder, so different members never write the same files. Each member also needs his or her own copy of ORCA in `MyDrive/Structural_Bioinformatics/ORCA`, as in the training.

Some calculations take long (see the **Planning the work** table of your group). Divide them so that everybody computes at the same time, each in his or her own Colab session, and start early: do not leave all the calculations for the last day.

If Colab disconnects, run the setup cells again and then the cell that was interrupted: the calculations already finished are not repeated. The free version of Colab disconnects after some time without activity, so keep the tab open and check it from time to time during the long cells.

# Setup (run these cells at the start of every session)

In [ ]:
GROUP = 1      # 1 (noncovalent interactions) or 2 (colour of chromophores)

In [ ]:
import os, glob, shutil, subprocess

ORCA_DIR = os.environ.get('ORCA_DIR', '')      # on your own computer: the folder containing the program orca
DRIVE_FOLDER = '/content/drive/MyDrive/Structural_Bioinformatics'

if not (ORCA_DIR and os.path.isfile(os.path.join(ORCA_DIR, 'orca'))):
    from google.colab import drive
    drive.mount('/content/drive')
    archives = sorted(glob.glob(os.path.join(DRIVE_FOLDER, 'ORCA', 'orca_6*linux_x86-64*.tar.*')))
    if not archives:
        raise FileNotFoundError('No ORCA archive found in ' + os.path.join(DRIVE_FOLDER, 'ORCA') +
                                ': follow the instructions above to download it.')
    archive = archives[-1]                        # if there is more than one, take the most recent version
    ORCA_DIR = os.path.join('/content', os.path.basename(archive).split('.tar')[0])
    if not os.path.isfile(os.path.join(ORCA_DIR, 'orca')):
        print('unpacking', os.path.basename(archive), '...')
        subprocess.run(['tar', '-xf', archive, '-C', '/content'], check=True)
    # some archives contain an additional folder level
    found = glob.glob(os.path.join('/content', 'orca_6*', '**', 'orca'), recursive=True)
    ORCA_DIR = os.path.dirname(sorted(found)[-1])

# ORCA must be called with its full path; the shared version also needs its libraries
os.environ['PATH'] = ORCA_DIR + ':' + os.environ['PATH']
os.environ['LD_LIBRARY_PATH'] = ORCA_DIR + ':' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['OMP_NUM_THREADS'] = '1'
ORCA = os.path.join(ORCA_DIR, 'orca')
version = subprocess.run([ORCA, '--version'], capture_output=True, text=True).stdout.strip()
print('ORCA found in', ORCA_DIR)
print(version if version else 'ORCA is installed')

In [ ]:
%pip install -q py3Dmol

The results are computed in a folder of the virtual machine and copied to the group folder `Task7_groupN` on Google Drive. The function `sync_from_drive()` brings to the virtual machine the results of the other members.

In [ ]:
if 'BASE_DIR' not in globals():          # remember where we started: running this cell again does not go one folder deeper
    BASE_DIR = os.getcwd()
if os.path.isdir('/content/drive/MyDrive'):          # on Colab: compute on the local disk, keep the results in the group folder
    local_path = f'/content/T07_group{GROUP}'
    DRIVE_PATH = f'/content/drive/MyDrive/Task7_group{GROUP}'
    os.makedirs(DRIVE_PATH, exist_ok=True)
else:                                                # on your computer: a folder next to the notebook
    local_path = os.path.join(BASE_DIR, f'T07_group{GROUP}')
    DRIVE_PATH = None
os.makedirs(local_path, exist_ok=True)

def sync_from_drive():
    # copy to the virtual machine the results of the whole group (also those computed by the other members)
    if DRIVE_PATH:
        shutil.copytree(DRIVE_PATH, local_path, dirs_exist_ok=True)

sync_from_drive()
%cd $local_path
print('results of the group are in', DRIVE_PATH if DRIVE_PATH else local_path)

# the input structures are in the Data folder of the course repository
if os.path.isdir(os.path.join(BASE_DIR, '..', 'Data', 'qm', 's22')):
    REPO_DIR = os.path.abspath(os.path.join(BASE_DIR, '..'))
else:
    REPO_DIR = '/content/sb' if os.path.isdir('/content') else os.path.join(BASE_DIR, 'sb')
    if os.path.isdir(REPO_DIR):
        !cd "$REPO_DIR" && git pull -q
    else:
        !git clone -q --depth 1 https://github.com/carlocamilloni/Structural-Bioinformatics.git "$REPO_DIR"
QM_DIR = os.path.join(REPO_DIR, 'Data', 'qm')
print('input files from', QM_DIR)

The helper functions of the training, and a few new ones (read them: they are short).

In [ ]:
import re, time
import numpy as np
import matplotlib.pyplot as plt
import py3Dmol

HARTREE2EV = 27.211386        # 1 Eh in eV
HARTREE2KCAL = 627.5095       # 1 Eh in kcal/mol
BOHR2ANG = 0.529177           # 1 bohr in Å

def save_to_drive(workdir):
    # copy the useful files of a calculation (not the large temporary ones) to the same folder on Google Drive
    if not DRIVE_PATH:
        return
    target = os.path.join(DRIVE_PATH, os.path.relpath(os.path.abspath(workdir), local_path))
    os.makedirs(target, exist_ok=True)
    for f in os.listdir(workdir):
        if f.endswith(('.inp', '.out', '.xyz', '.cube', '.csv', '.hess', '.json', '.png')):
            shutil.copy2(os.path.join(workdir, f), target)

def run_orca(name, inp, folder='.', rerun=False):
    # write folder/name/name.inp, run ORCA in folder/name and return the path of the output file
    workdir = os.path.join(folder, name)
    os.makedirs(workdir, exist_ok=True)
    out = os.path.join(workdir, name + '.out')
    inpfile = os.path.join(workdir, name + '.inp')
    inp = inp.strip() + '\n'
    if (not rerun and os.path.isfile(out) and 'ORCA TERMINATED NORMALLY' in open(out).read()
            and os.path.isfile(inpfile) and open(inpfile).read() == inp):
        print(f'{name}: already done, reading {out}')
        return out
    for f in os.listdir(workdir):            # a new run starts from an empty folder
        if os.path.isfile(os.path.join(workdir, f)):
            os.remove(os.path.join(workdir, f))
    with open(inpfile, 'w') as f:
        f.write(inp)
    t0 = time.time()
    with open(out, 'w') as f:
        subprocess.run([ORCA, name + '.inp'], cwd=workdir, stdout=f, stderr=subprocess.STDOUT)
    text = open(out).read()
    if 'ORCA TERMINATED NORMALLY' not in text:
        print(text[-3000:])
        raise RuntimeError(f'ORCA did not terminate normally: look at the end of {out}')
    print(f'{name}: done in {time.time() - t0:.1f} s')
    save_to_drive(workdir)
    return out

def xyz_block(xyzfile, ghost=()):
    # the coordinate lines of an xyz file; the atoms whose index is in ghost become ghost atoms (see Part 4)
    lines = open(xyzfile).read().splitlines()
    natoms = int(lines[0])
    atoms = []
    for i, l in enumerate(lines[2:2 + natoms]):
        el, x, y, z = l.split()[:4]
        atoms.append(f'{el}{":" if i in ghost else " "} {x:>12} {y:>12} {z:>12}')
    return '\n'.join(atoms)

def final_energy(out):
    # the last 'FINAL SINGLE POINT ENERGY' of an output (in Eh)
    return float(re.findall(r'FINAL SINGLE POINT ENERGY\s+(-?\d+\.\d+)', open(out).read())[-1])

def wall_time(out):
    # total run time of a calculation, in seconds
    d, h, m, s, ms = re.findall(r'TOTAL RUN TIME:\s+(\d+) days (\d+) hours (\d+) minutes (\d+) seconds (\d+) msec', open(out).read())[-1]
    return 86400*int(d) + 3600*int(h) + 60*int(m) + int(s) + int(ms)/1000

def s2(out):
    # expectation value of S^2 (only for UHF/UKS calculations)
    return float(re.findall(r'Expectation value of <S\*\*2>\s*:\s*(-?\d+\.\d+)', open(out).read())[-1])

def dipole(out):
    # magnitude of the dipole moment, in Debye
    return float(re.findall(r'Magnitude \(Debye\)\s*:\s*(\d+\.\d+)', open(out).read())[-1])

def orbital_energies(out):
    # the last table of orbital energies: list of (occupation, energy in eV); for UHF the spin-up orbitals
    text = open(out).read().split('ORBITAL ENERGIES')[-1]
    rows = []
    for l in text.splitlines()[4:]:
        f = l.split()
        if len(f) != 4:
            if rows: break
            continue
        rows.append((float(f[1]), float(f[3])))
    return rows

def charges(out, kind='MULLIKEN'):
    # atomic charges: kind is MULLIKEN, LOEWDIN, HIRSHFELD (needs the keyword Hirshfeld) or CHELPG (keyword CHELPG)
    text = open(out).read()
    if kind == 'MULLIKEN':
        block = text.split('MULLIKEN ATOMIC CHARGES')[-1].split('Sum of atomic charges')[0]
    elif kind == 'LOEWDIN':
        block = text.split('LOEWDIN ATOMIC CHARGES')[-1].split('LOEWDIN REDUCED')[0]
    elif kind == 'HIRSHFELD':
        block = text.split('HIRSHFELD ANALYSIS')[-1].split('TOTAL')[0]
    elif kind == 'CHELPG':
        block = text.split('CHELPG Charges')[-1].split('Total charge')[0]
    return [(el, float(q)) for el, q in re.findall(r'^\s*\d+\s+([A-Z][a-z]?)\s*:?\s+(-?\d+\.\d+)', block, re.M)]

def read_xyz(xyzfile):
    lines = open(xyzfile).read().splitlines()
    n = int(lines[0])
    el = [l.split()[0] for l in lines[2:2+n]]
    xyz = np.array([[float(v) for v in l.split()[1:4]] for l in lines[2:2+n]])
    return el, xyz

def read_cube(cubefile):
    # Gaussian cube file -> (origin, axes, values); origin and axes in bohr
    with open(cubefile) as f:
        f.readline(); f.readline()
        n, *origin = f.readline().split()
        natoms = abs(int(n))
        axes, shape = [], []
        for _ in range(3):
            npt, *v = f.readline().split()
            shape.append(int(npt)); axes.append([float(x) for x in v])
        for _ in range(natoms):
            f.readline()
        data = np.array(f.read().split(), dtype=float)
    return np.array(origin, dtype=float), np.array(axes), data[-np.prod(shape):].reshape(shape)

def show_molecule(xyzfile, width=400, height=300):
    view = py3Dmol.view(width=width, height=height)
    view.addModel(open(xyzfile).read(), 'xyz')
    view.setStyle({'stick': {'radius': 0.15}, 'sphere': {'scale': 0.25}})
    view.zoomTo()
    return view.show()

def cube_for_display(cubefile, maxpoints=40):
    # the text of a cube file with at most maxpoints per side and 4 digits: much smaller in the notebook
    with open(cubefile) as f:
        lines = f.readlines()
    natoms = abs(int(lines[2].split()[0]))
    origin, axes, data = read_cube(cubefile)
    step = [max(1, int(np.ceil(n / maxpoints))) for n in data.shape]
    data = data[::step[0], ::step[1], ::step[2]]
    head = lines[:2] + [lines[2]]
    for i in range(3):
        v = axes[i] * step[i]
        head.append(f'{data.shape[i]:5d} {v[0]:12.6f} {v[1]:12.6f} {v[2]:12.6f}\n')
    head += lines[6:6 + natoms]
    values = data.ravel()
    body = '\n'.join(' '.join(f'{x:.4g}' for x in values[k:k + 6]) for k in range(0, len(values), 6))
    return ''.join(head) + body + '\n'

def cube_atoms_xyz(cubefile):
    # the atoms stored in a cube file (in bohr) as an xyz text (in Å)
    elements = {1: 'H', 6: 'C', 7: 'N', 8: 'O', 16: 'S'}
    with open(cubefile) as f:
        lines = f.readlines()
    natoms = abs(int(lines[2].split()[0]))
    atoms = []
    for l in lines[6:6 + natoms]:
        z, _, x, y, w = l.split()[:5]
        atoms.append(f'{elements.get(int(float(z)), "X")} {float(x) * BOHR2ANG:.4f} {float(y) * BOHR2ANG:.4f} {float(w) * BOHR2ANG:.4f}')
    return f'{natoms}\n\n' + '\n'.join(atoms) + '\n'

def show_cube(cubefile, isovalue=0.05, signed=False, width=400, height=300):
    # isosurface of a cube file; signed=True for orbitals (positive lobes red, negative lobes blue)
    view = py3Dmol.view(width=width, height=height)
    view.addModel(cube_atoms_xyz(cubefile), 'xyz')
    view.setStyle({'stick': {'radius': 0.12}, 'sphere': {'scale': 0.2}})
    data = cube_for_display(cubefile)
    view.addVolumetricData(data, 'cube', {'isoval': isovalue, 'color': 'red', 'opacity': 0.8})
    if signed:
        view.addVolumetricData(data, 'cube', {'isoval': -isovalue, 'color': 'blue', 'opacity': 0.8})
    view.zoomTo()
    return view.show()

def plots_block(cubes, npoints=50):
    # an ORCA %plots block that writes cube files; cubes is a list of strings like
    # 'ElDens("x.eldens.cube")' or 'MO("x.mo4.cube",4,0)' (orbitals are numbered from 0)
    return '%plots\n  dim1 {0}\n  dim2 {0}\n  dim3 {0}\n  Format Gaussian_Cube\n'.format(npoints) + \
           ''.join(f'  {c};\n' for c in cubes) + 'end\n'

In [ ]:
def interaction_energy(xyzfile, nA, method, folder, counterpoise=True, q=(0, 0, 0)):
    # interaction energy (kcal/mol) of a dimer whose first nA atoms are molecule A and the others molecule B
    # q = (charge of AB, charge of A, charge of B); all species are closed-shell singlets
    nat = int(open(xyzfile).readline())
    tag = os.path.basename(xyzfile).replace('.xyz', '') + '_' + re.sub(r'[^A-Za-z0-9_+-]', '', method.replace(' ', '_'))   # no spaces, / or () in file names
    A, B = range(nA), range(nA, nat)
    jobs = {'AB': (xyz_block(xyzfile), q[0])}
    if counterpoise:                                     # monomers in the basis of the dimer: the partner is made of ghost atoms
        jobs['A'] = (xyz_block(xyzfile, ghost=B), q[1])
        jobs['B'] = (xyz_block(xyzfile, ghost=A), q[2])
    else:                                                # monomers in their own basis: the partner is removed
        lines = xyz_block(xyzfile).splitlines()
        jobs['A'] = ('\n'.join(lines[:nA]), q[1])
        jobs['B'] = ('\n'.join(lines[nA:]), q[2])
    E = {}
    for k, (coords, charge) in jobs.items():
        name = f'{tag}_{k}' + ('_cp' if counterpoise and k != 'AB' else '')
        out = run_orca(name, f'! {method} TightSCF\n* xyz {charge} 1\n{coords}\n*', folder=folder)
        E[k] = final_energy(out)
    return (E['AB'] - E['A'] - E['B']) * HARTREE2KCAL

In [ ]:
def excitations(out):
    # excited states from the absorption spectrum table: list of (energy eV, wavelength nm, oscillator strength)
    text = open(out).read().split('ABSORPTION SPECTRUM VIA TRANSITION ELECTRIC DIPOLE MOMENTS')[-1].split('\n\n')[0]
    rows = []
    for l in text.splitlines():
        if '->' in l:
            f = l.split('->')[1].split()
            rows.append((float(f[1]), float(f[3]), float(f[4])))
    return np.array(rows)

def transitions(out):
    # the orbital contributions to each excited state, as printed by ORCA (e.g. "7a -> 8a : 0.98")
    text = open(out).read().split('EXCITED STATES')[-1].split('ABSORPTION SPECTRUM')[0]
    for block in re.split(r'\n(?=STATE\s+\d+:)', text)[1:]:
        head = block.splitlines()[0]
        contr = re.findall(r'(\d+a\s+->\s+\d+a)\s+:\s+(\d\.\d+)', block)
        print(head.split('<S**2>')[0].strip(), '  ', ',  '.join(f'{t} ({c})' for t, c in contr))

def spectrum(states, fwhm=0.3, emin=1.5, emax=8.0):
    # absorption spectrum: a Gaussian of width fwhm (eV) for each state, with area proportional to f
    E = np.linspace(emin, emax, 1000)
    sigma = fwhm / 2.3548
    eps = sum(f * np.exp(-(E - e)**2 / (2 * sigma**2)) / (sigma * np.sqrt(2 * np.pi)) for e, lam, f in states)
    return E, eps

def colour_of(states, fwhm=0.3, max_absorbance=1.0):
    # colour of the light transmitted by a solution of the molecule (white light in, absorbed bands removed),
    # with the concentration chosen so that the strongest band has absorbance max_absorbance
    lam = np.arange(380.0, 781.0, 1.0)                        # visible wavelengths (nm)
    def g(x, mu, s1, s2):                                    # CIE 1931 colour matching functions (Wyman et al. 2013)
        return np.exp(-0.5 * ((x - mu) / np.where(x < mu, s1, s2))**2)
    xbar = 1.056 * g(lam, 599.8, 37.9, 31.0) + 0.362 * g(lam, 442.0, 16.0, 26.7) - 0.065 * g(lam, 501.1, 20.4, 26.2)
    ybar = 0.821 * g(lam, 568.8, 46.9, 40.5) + 0.286 * g(lam, 530.9, 16.3, 31.1)
    zbar = 1.217 * g(lam, 437.0, 11.8, 36.0) + 0.681 * g(lam, 459.0, 26.0, 13.8)
    sigma = fwhm / 2.3548
    E = 1239.84 / lam
    band = lambda e: np.exp(-(E - e)**2 / (2 * sigma**2))
    E_all = np.linspace(1.0, 12.0, 4000)
    peak = max(sum(f * np.exp(-(E_all - e)**2 / (2 * sigma**2)) for e, _, f in states).max(), 1e-12)
    absorbance = max_absorbance * sum(f * band(e) for e, _, f in states) / peak
    T = 10**(-absorbance)
    XYZ = np.array([np.sum(T * c) for c in (xbar, ybar, zbar)]) / np.sum(ybar)
    M = np.array([[3.2406, -1.5372, -0.4986], [-0.9689, 1.8758, 0.0415], [0.0557, -0.2040, 1.0570]])
    white = M @ (np.array([np.sum(c) for c in (xbar, ybar, zbar)]) / np.sum(ybar))
    rgb = np.clip(M @ XYZ / white, 0, 1)                     # white light -> (1, 1, 1)
    rgb = np.where(rgb <= 0.0031308, 12.92 * rgb, 1.055 * rgb**(1 / 2.4) - 0.055)
    return np.clip(rgb, 0, 1)

In [ ]:
import json, glob, linecache, warnings
import pandas as pd
from IPython import get_ipython
from IPython.display import display
from IPython.core.magic import register_cell_magic
warnings.filterwarnings('ignore')

@register_cell_magic
def group(line, cell):
    # a cell that starts with %%group N runs only if GROUP is N
    ipython = get_ipython()
    if int(line) != ipython.user_ns.get('GROUP'):
        print(f'This cell is for group {line}: skipped.')
        return
    code = ipython.transform_cell(cell)
    filename = f'<group {line} cell>'
    linecache.cache[filename] = (len(code), None, code.splitlines(True), filename)   # so that errors show the code
    exec(compile(code, filename, 'exec'), ipython.user_ns)

def save_figure(fig, name):
    # save a matplotlib figure for your presentation, in the folder figures of the group (also on Google Drive)
    os.makedirs(os.path.join(local_path, 'figures'), exist_ok=True)
    fig.savefig(os.path.join(local_path, 'figures', name + '.png'), dpi=200, bbox_inches='tight')
    save_file_to_drive(os.path.join(local_path, 'figures', name + '.png'))
    print('saved figures/' + name + '.png')

def save_file_to_drive(filename):
    # copy only this file to the group folder (copying the whole folder could overwrite newer results of the others)
    if DRIVE_PATH:
        target = os.path.join(DRIVE_PATH, os.path.relpath(os.path.abspath(filename), local_path))
        os.makedirs(os.path.dirname(target), exist_ok=True)
        shutil.copy2(filename, target)

def save_json(data, filename):
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    with open(filename, 'w') as f:
        json.dump(data, f, indent=1)
    save_file_to_drive(filename)

def dispersion(out):
    # the D3/D4 dispersion correction (Eh) included in the final energy; 0 if there is none
    v = re.findall(r'Dispersion correction\s+(-?\d+\.\d+)', open(out).read())
    return float(v[-1]) if v else 0.0

def interaction_energy_split(xyzfile, nA, method, folder, q=(0, 0, 0)):
    # counterpoise-corrected interaction energy (kcal/mol), split in two parts: without the dispersion
    # correction, and the dispersion correction (so one B3LYP-D4 calculation gives both B3LYP and B3LYP-D4)
    nat = int(open(xyzfile).readline())
    tag = os.path.basename(xyzfile).replace('.xyz', '') + '_' + re.sub(r'[^A-Za-z0-9_+-]', '', method.replace(' ', '_'))
    A, B = range(nA), range(nA, nat)
    E, D = {}, {}
    for k, ghost, charge in [('AB', (), q[0]), ('A', B, q[1]), ('B', A, q[2])]:
        name = f'{tag}_{k}' + ('_cp' if k != 'AB' else '')
        out = run_orca(name, f'! {method} TightSCF\n* xyz {charge} 1\n{xyz_block(xyzfile, ghost=ghost)}\n*', folder=folder)
        E[k], D[k] = final_energy(out), dispersion(out)
    total = (E['AB'] - E['A'] - E['B']) * HARTREE2KCAL
    disp = (D['AB'] - D['A'] - D['B']) * HARTREE2KCAL
    return total - disp, disp

def show_pdb_context(pdb, highlights, title='', labels=(), lines=(), local_file=None, width=700, height=420):
    # a structure from the PDB (downloaded by your browser from rcsb.org) or from a local file, with some residues
    # highlighted as sticks; highlights: list of 3Dmol selections, e.g. {'chain': 'A', 'resi': 6}
    if local_file:
        view = py3Dmol.view(width=width, height=height)
        view.addModel(open(local_file).read(), 'pdb')
    else:
        view = py3Dmol.view(query='pdb:' + pdb, width=width, height=height)
    view.setStyle({'cartoon': {'color': 'lightgrey', 'opacity': 0.8}})
    view.addStyle({'resn': ['DA', 'DT', 'DG', 'DC', 'A', 'U', 'G', 'C']}, {'stick': {'radius': 0.08, 'color': 'lightgrey'}})
    for k, sel in enumerate(highlights):
        view.addStyle(sel, {'stick': {'radius': 0.22, 'colorscheme': ['greenCarbon', 'orangeCarbon', 'cyanCarbon', 'magentaCarbon'][k % 4]}})
    for sel in labels:
        view.addResLabels(sel, {'fontSize': 11, 'showBackground': True, 'backgroundOpacity': 0.6})
    for p, q in lines:
        view.addCylinder({'start': dict(zip('xyz', map(float, p))), 'end': dict(zip('xyz', map(float, q))),
                          'radius': 0.1, 'dashed': True, 'color': 'darkorange'})
    view.zoomTo({'or': highlights} if len(highlights) > 1 else highlights[0] if highlights else {})
    print(title)
    return view.show()

# GROUP 1, Part 1: running the calculations

The structures of the group are listed in `Data/qm/interactions/interactions.csv`. In each xyz file the atoms of molecule A come first (`nA` atoms), then those of molecule B; `qA` and `qB` are their charges.

In [ ]:
%%group 1
INTER_DIR = os.path.join(QM_DIR, 'interactions')
catalogue = pd.read_csv(os.path.join(INTER_DIR, 'interactions.csv')).set_index('name')
display(catalogue[['family', 'description', 'biology', 'qA', 'qB', 'reference_kcal']])

def complex_info(name):
    row = catalogue.loc[name]
    return os.path.join(INTER_DIR, name + '.xyz'), int(row.nA), int(row.qA), int(row.qB)

SOLVENT = {'xtb': 'ALPB(water)', 'dft': 'CPCM(water)'}   # continuum water for the two kinds of methods

def interaction(name, method, solvent=False, coords=None, tag=''):
    # interaction energy E(AB) - E(A) - E(B) in kcal/mol, and its dispersion part (0 for GFN2-xTB, which does not print it)
    xyzfile, nA, qA, qB = complex_info(name)
    el, x = read_xyz(xyzfile) if coords is None else coords
    keywords = f'! {method} TightSCF' if 'XTB' not in method else f'! {method}'
    if solvent:
        keywords += ' ' + SOLVENT['xtb' if 'XTB' in method else 'dft']
    folder = os.path.join('g1', name, re.sub(r'[^A-Za-z0-9_+-]', '', method) + ('_water' if solvent else '_vacuum'))
    E, D = {}, {}
    for k, idx, q in [('AB', range(len(el)), qA + qB), ('A', range(nA), qA), ('B', range(nA, len(el)), qB)]:
        block = '\n'.join(f'{el[i]} {x[i, 0]:.5f} {x[i, 1]:.5f} {x[i, 2]:.5f}' for i in idx)
        out = run_orca(f'{k}{tag}', f'{keywords}\n* xyz {q} 1\n{block}\n*', folder=folder)
        E[k], D[k] = final_energy(out), dispersion(out)
    total = (E['AB'] - E['A'] - E['B']) * HARTREE2KCAL
    disp = (D['AB'] - D['A'] - D['B']) * HARTREE2KCAL
    return total, disp

for name in ['uracil_hbond', 'uracil_stack']:
    print(catalogue.loc[name, 'description'])
    show_molecule(complex_info(name)[0])

## Step 1: choosing the methods

One member runs these cells (about 5 minutes), the others will find the results in the group folder. For each method the cell prints the interaction energy of the two uracil complexes, the error with respect to the reference, and the computer time.

*Optional*: add the more accurate (and about ten times slower) DFT method `'wB97X-3c'` to the list, or another method you are curious about.

In [ ]:
%%group 1
CANDIDATES = ['Native-GFN2-XTB', 'r2SCAN-3c']
rows = []
for name in ['uracil_hbond', 'uracil_stack']:
    ref = catalogue.loc[name, 'reference_kcal']
    for method in CANDIDATES:
        e, d = interaction(name, method)
        # computer time of the three calculations, as reported by ORCA
        folder = os.path.join('g1', name, re.sub(r'[^A-Za-z0-9_+-]', '', method) + '_vacuum')
        dt = sum(wall_time(os.path.join(folder, k, k + '.out')) for k in ['AB', 'A', 'B'])
        rows.append({'complex': name, 'method': method, 'energy': e, 'error': e - ref, 'time (s)': dt})
        if d:
            rows.append({'complex': name, 'method': method + ' without dispersion', 'energy': e - d, 'error': e - d - ref, 'time (s)': dt})
calibration = pd.DataFrame(rows)
save_json(calibration.to_dict(orient='list'), os.path.join('g1', 'results', 'calibration.json'))
display(calibration.round(2))

From now on we use two methods: change them here if the calibration (yours, or the discussion in the group) suggests a different choice.

In [ ]:
%%group 1
FAST = 'Native-GFN2-XTB'     # for the scans (many points)
ACCURATE = 'r2SCAN-3c'       # for the final numbers

## Step 2: your complexes

Write the names of your complexes (see the table of the assignment: one family or more) and run the cells below. Whoever ran Step 1 also sets `ALSO_URACIL = True`, so that the two uracil complexes appear in the results of the group.

In [ ]:
%%group 1
my_complexes = ['at_wc', 'gc_wc']
ALSO_URACIL = False      # True for whoever ran Step 1: the two uracil complexes are added to the interaction map
if ALSO_URACIL:
    my_complexes = my_complexes + ['uracil_hbond', 'uracil_stack']

for name in my_complexes:
    print(name, ':', catalogue.loc[name, 'description'], '-', catalogue.loc[name, 'biology'])
    show_molecule(complex_info(name)[0])

### The electrostatic potential: where are the positive and negative parts?

To see the electrostatics, we compute for each molecule (alone, at the geometry it has in the complex) the electron density and the **electrostatic potential** (ESP): the energy that a positive charge would feel at each point of space. The ESP is drawn with colours on the surface of the molecule (where the density is 0.002 electrons/bohr$^3$, close to the van der Waals surface): **red** where it is negative (electron-rich: lone pairs, π clouds of aromatic rings, negative ions), **blue** where it is positive (electron-poor: H atoms bound to N or O, positive ions). The program `orca_vpot` computes it on the points of the density grid.

In [ ]:
%%group 1
def esp_cubes(name, coords, charge, folder, npoints=50):
    # density and electrostatic potential (cube files) of a molecule, and its output
    workdir = os.path.join(folder, name)
    dens, esp = os.path.join(workdir, name + '.dens.cube'), os.path.join(workdir, name + '.esp.cube')
    inp = f'! {ACCURATE} CHELPG TightSCF KeepDens\n' + plots_block([f'ElDens("{name}.dens.cube")'], npoints) + \
          f'* xyz {charge} 1\n{coords}\n*'
    rerun = not os.path.isfile(esp) and not os.path.isfile(os.path.join(workdir, name + '.gbw'))
    out = run_orca(name, inp, folder=folder, rerun=rerun)
    if not os.path.isfile(esp):
        lines = open(dens).read().splitlines()
        natoms = abs(int(lines[2].split()[0]))
        origin, axes, rho = read_cube(dens)
        points = origin + np.indices(rho.shape).reshape(3, -1).T @ axes
        np.savetxt(os.path.join(workdir, 'points.txt'), points, header=str(len(points)), comments='')
        subprocess.run([os.path.join(ORCA_DIR, 'orca_vpot'), name + '.gbw', name + '.scfp', 'points.txt', 'esp.txt'],
                       cwd=workdir, capture_output=True)
        v = np.loadtxt(os.path.join(workdir, 'esp.txt'), skiprows=1)[:, 3]
        body = '\n'.join(' '.join(f'{x:.5e}' for x in v[k:k + 6]) for k in range(0, len(v), 6))
        with open(esp, 'w') as f:
            f.write('\n'.join(lines[:6 + natoms]) + '\n' + body + '\n')
        save_to_drive(workdir)
    return out, dens, esp

def show_esp(xyzfile, surfaces, isovalue=0.002, vmax=0.04, width=500, height=400):
    # molecules as sticks, with the ESP (red negative, blue positive, in Eh/e) on the density surface
    # (for ions the colours saturate: increase vmax, e.g. to 0.15, to see the details)
    view = py3Dmol.view(width=width, height=height)
    view.addModel(open(xyzfile).read(), 'xyz')
    view.setStyle({'stick': {'radius': 0.12}})
    for dens, esp in surfaces:
        view.addVolumetricData(cube_for_display(dens, 32), 'cube',
                               {'isoval': isovalue, 'opacity': 0.75, 'voldata': cube_for_display(esp, 32), 'volformat': 'cube',
                                'volscheme': {'gradient': 'rwb', 'min': -vmax, 'max': vmax}})
    view.zoomTo()
    return view.show()

### All the calculations of a complex

The next cell runs, for each of your complexes:

1.  the interaction energy with the accurate method, in vacuum and in water (as a continuum), with its dispersion part;
2.  the interaction energy with the fast method as a function of the distance: molecule B is moved rigidly along the line joining the centres of the two molecules, from 0.3 Å closer to 4 Å further than in the optimised complex, in vacuum and in water;
3.  the density and the ESP of the two molecules.

The results are collected in `g1/results/<complex>.json`. If Colab disconnects, run the cell again: what is already done is not repeated.

In [ ]:
%%group 1
SHIFTS = [-0.3, -0.15, 0.0, 0.25, 0.5, 1.0, 1.5, 2.5, 4.0]     # displacement of molecule B (Å)

def run_complex(name):
    t0 = time.time()
    xyzfile, nA, qA, qB = complex_info(name)
    el, x = read_xyz(xyzfile)
    res = {'name': name, 'family': catalogue.loc[name, 'family'], 'biology': catalogue.loc[name, 'biology'],
           'reference': None if pd.isna(catalogue.loc[name, 'reference_kcal']) else float(catalogue.loc[name, 'reference_kcal'])}
    # 1. accurate method, vacuum and water
    for solvent in [False, True]:
        e, d = interaction(name, ACCURATE, solvent)
        res['water' if solvent else 'vacuum'] = {'energy': e, 'dispersion': d}
    # 2. scan with the fast method: B moves along the line joining the centres of the two molecules
    direction = x[nA:].mean(axis=0) - x[:nA].mean(axis=0)
    direction /= np.linalg.norm(direction)
    res['scan'] = {'shift': SHIFTS, 'distance': [], 'vacuum': [], 'water': []}
    for s in SHIFTS:
        y = x.copy()
        y[nA:] += s * direction
        res['scan']['distance'].append(float(np.linalg.norm(y[nA:].mean(axis=0) - y[:nA].mean(axis=0))))
        for solvent in [False, True]:
            try:
                e, _ = interaction(name, FAST, solvent, coords=(el, y), tag=f'_{s:+.2f}')
            except RuntimeError:          # the fast method does not converge (see section 3.5): the point is left out
                print(f'   {FAST} failed at {s:+.2f} Å ({"water" if solvent else "vacuum"}): point left out')
                e = float('nan')
            res['scan']['water' if solvent else 'vacuum'].append(e)
    # 3. density and ESP of the two molecules
    lines = xyz_block(xyzfile).splitlines()
    for k, coords, q in [('A', lines[:nA], qA), ('B', lines[nA:], qB)]:
        esp_cubes(f'monomer_{k}', '\n'.join(coords), q, os.path.join('g1', name))
    save_json(res, os.path.join('g1', 'results', name + '.json'))
    print(f'{name} done in {(time.time() - t0) / 60:.1f} minutes\n')

for name in my_complexes:
    run_complex(name)

A first look at your results:

In [ ]:
%%group 1
for name in my_complexes:
    res = json.load(open(os.path.join('g1', 'results', name + '.json')))
    v, w = res['vacuum'], res['water']
    print(f"{name}: {ACCURATE} vacuum {v['energy']:.2f} kcal/mol (dispersion {v['dispersion']:.2f}), water {w['energy']:.2f} kcal/mol"
          + (f"; reference (vacuum) {res['reference']:.2f}" if res['reference'] is not None else ''))

# GROUP 1, Part 2: the report

The cells of the report read the results of all the members of the group (run them again when the others have finished: `sync_from_drive()` brings their results here).

## 1. Introduction

*Explain to someone who has not studied this topic: what noncovalent interactions are, why they matter for the structure and function of biomolecules, and what kinds there are. Which interactions did your group study, and where are they found in DNA, RNA and proteins? Why are quantum calculations useful here, given that MD simulations use force fields?*

*Your text here.*

## 2. Methods

*Describe in a few sentences, without equations, what you computed and how: where the structures come from; how an interaction energy is computed; the two methods you chose and why (Step 1); how water was described; the distance scans; the electrostatic potential.*

*Your text here.*

## 3. Results

In [ ]:
%%group 1
sync_from_drive()
results = {}
for f in sorted(glob.glob(os.path.join('g1', 'results', '*.json'))):
    name = os.path.basename(f)[:-5]
    if name == 'calibration':
        continue
    res = json.load(open(f))
    if name in catalogue.index and res.get('name') == name:
        results[name] = res
    else:              # e.g. a copy made by Google Drive ('at_wc (1).json'), or a complex no longer in the catalogue
        print('ignored:', f)
print('complexes of the group:', list(results))
FAMILY_COLOURS = {'calibration': 'grey', 'nucleic acids': 'C0', 'aromatic contacts': 'C1', 'charged groups': 'C3', 'protein–DNA recognition': 'C2'}

### 3.1 Choosing the methods

**Table 1**: interaction energies, errors and computer times of the candidate methods for the two uracil complexes (Step 1).

*How accurate is each method for the hydrogen-bonded and for the stacked complex? Which part of the interaction is lost without the dispersion correction, and for which complex does it matter most? Which methods did you choose as fast and accurate, and why? Is a good result on two complexes enough to trust a method? (Section 3.5 gives a partial answer.)*

In [ ]:
%%group 1
calibration = pd.DataFrame(json.load(open(os.path.join('g1', 'results', 'calibration.json'))))
display(calibration.round(2))
fig, ax = plt.subplots(figsize=(6, 4))
for (cplx, m), d in calibration.groupby(['complex', 'method']):
    ax.scatter(d['time (s)'], d['error'].abs(), marker='o' if 'hbond' in cplx else 's', s=60,
               color=f"C{list(calibration.method.unique()).index(m)}", label=f'{m}, {cplx}')
ax.set_xscale('log'); ax.axhline(1, ls='--', color='grey', lw=0.8)
ax.set_xlabel('computer time for one interaction energy (s)'); ax.set_ylabel('|error| (kcal/mol)'); ax.legend(fontsize=8)
plt.tight_layout(); save_figure(fig, 'g1_calibration'); plt.show()

*Your text here.*

### 3.2 The complexes and their electrostatic potential

**Figure 1** (one per complex, screenshot): each complex with the ESP of the two molecules. Rotate it to show the contact and take a screenshot. To see one molecule at a time, keep only one element of the list `surfaces`; for the charged complexes, try a larger `vmax`.

*Describe the figures: where are the red and blue regions of each molecule? In the hydrogen-bonded complexes, which regions face each other? And in the stacked ones? What is special about the charged complexes?*

In [ ]:
%%group 1
for name in results:
    xyzfile, nA, qA, qB = complex_info(name)
    folder = os.path.join('g1', name)
    surfaces = [(f'{folder}/monomer_{k}/monomer_{k}.dens.cube', f'{folder}/monomer_{k}/monomer_{k}.esp.cube') for k in 'AB']
    print(name, ':', catalogue.loc[name, 'biology'])
    show_esp(xyzfile, surfaces, vmax=0.04 if qA == qB == 0 else 0.1)

*Your text here.*

### 3.3 The interaction map

**Figure 2**: the interaction energy of all the complexes of the group (accurate method), in vacuum and in water. Points below the diagonal are interactions weakened by water.

*Rank the interactions by strength in vacuum, and then in water: does the ranking change? Which interactions are weakened most by water, and why (think of the ESP pictures and of what water does to charges)? Compare base pairing and stacking, salt bridges and cation–π, neutral and charge-assisted hydrogen bonds. What does this suggest about the interactions that really stabilise proteins and DNA in the cell?*

In [ ]:
%%group 1
table = pd.DataFrame({name: {'family': r['family'], 'biology': r['biology'], 'vacuum': r['vacuum']['energy'],
                             'water': r['water']['energy'], 'dispersion (vacuum)': r['vacuum']['dispersion'], 'reference': r['reference']}
                      for name, r in results.items()}).T
display(table)
fig, axs = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, zoom in zip(axs, [False, True]):
    strong = table[table['vacuum'] < -35].sort_values('vacuum').index    # in the full map, only these are labelled (with arrows)
    for name, r in table.iterrows():
        if zoom and r['vacuum'] < -35:
            continue
        ax.scatter(r['vacuum'], r['water'], s=70, color=FAMILY_COLOURS.get(r['family'], 'k'), zorder=3)
        if zoom:
            ax.annotate(name, (r['vacuum'], r['water']), fontsize=8, xytext=(5, 3), textcoords='offset points')
        elif name in strong:
            k = list(strong).index(name)
            ax.annotate(name, (r['vacuum'], r['water']), fontsize=8, xytext=(r['vacuum'] + 15, -35 - 12 * k),
                        arrowprops=dict(arrowstyle='-', color='grey', lw=0.6))
    lo = -35 if zoom else min(table['vacuum'].min(), table['water'].min()) - 5
    ax.plot([lo, 0], [lo, 0], 'k--', lw=0.8); ax.axhline(0, color='grey', lw=0.5); ax.axvline(0, color='grey', lw=0.5)
    ax.set_xlabel('interaction energy in vacuum (kcal/mol)'); ax.set_ylabel('interaction energy in water (kcal/mol)')
    ax.set_title('zoom on the weaker interactions' if zoom else f'interaction map ({ACCURATE})', loc='left')
for fam, col in FAMILY_COLOURS.items():
    axs[0].scatter([], [], color=col, label=fam)
axs[0].legend(fontsize=8, loc='lower right')
plt.tight_layout(); save_figure(fig, 'g1_interaction_map'); plt.show()

*Your text here.*

### 3.4 Where does the binding come from?

The dispersion correction of the accurate method is a separate term, so we can split the interaction energy (in vacuum) into dispersion and everything else (electrostatics, polarisation and Pauli repulsion).

**Figure 3**: the two contributions for all the complexes of the group.

*Which complexes are held together mainly by dispersion? Is dispersion negligible for the hydrogen bonds and for the charged complexes, in absolute value and relative to the total? Relate the result to the shape of the molecules (flat aromatic rings, polar groups, charges).*

In [ ]:
%%group 1
names = list(table.index)
total = table['vacuum'].astype(float).values
disp = table['dispersion (vacuum)'].astype(float).values
fig, ax = plt.subplots(figsize=(1.2 * len(names) + 3, 4.5))
x = np.arange(len(names))
ax.bar(x, total - disp, width=0.6, label='electrostatics, polarisation, repulsion')
ax.bar(x, disp, width=0.6, bottom=np.minimum(total - disp, 0), label='dispersion')
ax.set_xticks(x); ax.set_xticklabels(table['biology'], rotation=30, ha='right')
ax.axhline(0, color='grey', lw=0.8); ax.set_ylabel('interaction energy in vacuum (kcal/mol)'); ax.legend(fontsize=9)
plt.tight_layout(); save_figure(fig, 'g1_dispersion'); plt.show()
for n, t, d in zip(names, total, disp):
    print(f'{n:28s} dispersion = {d:6.2f} kcal/mol, {100 * d / t:4.0f}% of the total')

*Your text here.*

### 3.5 Pulling the molecules apart

**Figure 4**: the interaction energy as a function of the distance between the centres of the two molecules, computed with the fast method in vacuum and in water. The points of the accurate method at the optimised distance are shown for comparison.

For the salt bridge in vacuum, GFN2-xTB cannot describe the two ions far apart: its calculation does not converge, because in this approximate method an electron would jump from the acetate to the guanidinium, and the missing points are left out. This is a limit of the fast method worth discussing.

*How quickly do the different interactions decay with the distance? Which ones are still felt at 10 Å, and why (think of how the energy between two charges, between two dipoles and between two fluctuating electron clouds decays with the distance)? How does water change the shape of the curves? How well does the fast method agree with the accurate one: is it good enough for all types of interaction?*

In [ ]:
%%group 1
n = len(results)
ncol = min(n, 4)
fig, axs = plt.subplots((n + ncol - 1) // ncol, ncol, figsize=(4.2 * ncol, 3.6 * ((n + ncol - 1) // ncol)), squeeze=False)
for ax, (name, r) in zip(axs.flat, results.items()):
    sc = r['scan']
    ax.plot(sc['distance'], sc['vacuum'], 'o-', ms=4, label=f'{FAST}, vacuum')
    ax.plot(sc['distance'], sc['water'], 's--', ms=4, label=f'{FAST}, water')
    d0 = sc['distance'][sc['shift'].index(0.0)]
    ax.plot([d0], [r['vacuum']['energy']], 'k*', ms=12, label=f'{ACCURATE}, vacuum')
    ax.plot([d0], [r['water']['energy']], 'kX', ms=9, label=f'{ACCURATE}, water')
    ax.axhline(0, c='grey', lw=0.6); ax.set_title(r['biology'], fontsize=10)
    ax.set_xlabel('distance between the centres (Å)')
for ax in axs.flat[n:]:
    ax.axis('off')
axs[0, 0].set_ylabel('interaction energy (kcal/mol)'); axs[0, 0].legend(fontsize=7)
plt.tight_layout(); save_figure(fig, 'g1_curves'); plt.show()

*Your text here.*

### 3.6 Back to the real biomolecules

Our complexes are small models. The cells below show where the same interactions occur in real structures: a DNA double helix (PDB 1BNA), a transfer RNA (1EHZ), a protein kinase (the Abl kinase of the docking practical, 2GQG) and a protein bound to DNA (the DNA-binding domain of the estrogen receptor, 1HCQ). The structures from the PDB are downloaded by your browser from the PDB; if they don't appear, download the file from [rcsb.org](https://www.rcsb.org) and open it in VMD or ChimeraX.

**Figure 5** (screenshot): your interactions in their biological context.

*Compare the arrangement in the real structure with your model complex (distances, orientation: you can measure them in VMD or ChimeraX). How many interactions of your type are there in the structure? Proteins that bind DNA use arginine to hold the phosphates more often than lysine (Luscombe et al. 2001): can your results explain why? Using your energies (in water), estimate their total contribution to the stability of the structure, and compare it with typical folding free energies (5–15 kcal/mol for a small protein). What does the comparison tell you?*

In [ ]:
%%group 1
KINASE = os.path.join(REPO_DIR, 'Data', 'docking', '2gqg.pdb')

def backbone_hbonds(pdbfile, cutoff=3.1):
    # backbone N...O pairs closer than cutoff (Å), between residues at least 3 apart: the H-bonds of helices and sheets
    N, O = {}, {}
    for l in open(pdbfile):
        if l.startswith('ATOM') and l[12:16].strip() in ('N', 'O'):
            (N if l[12:16].strip() == 'N' else O)[int(l[22:26])] = np.array([float(l[30:38]), float(l[38:46]), float(l[46:54])])
    return [(N[i], O[j]) for i in N for j in O if abs(i - j) > 2 and np.linalg.norm(N[i] - O[j]) < cutoff]

def kinase(res, title):
    return dict(local_file=KINASE, highlights=[{'resi': res}], labels=[{'resi': res}], title=f'Abl kinase (2GQG): {title}')

# the DNA-binding domain of the estrogen receptor on its DNA: Arg56, Lys57 and Arg63 hold the same phosphate (atom pairs closer than 3.2 Å)
dna_binding = dict(pdb='1HCQ', highlights=[{'chain': 'A', 'resi': [56, 63]}, {'chain': 'A', 'resi': 57}, {'chain': 'D', 'resi': [30, 31]}, {'chain': 'A', 'resi': 33}],
                   labels=[{'chain': 'A', 'resi': [33, 56, 57, 63]}],
                   lines=[((40.500, 15.882, 102.098), (40.508, 13.480, 103.681)),     # Arg56 NE - G31 OP1
                          ((42.070, 14.385, 105.688), (40.508, 13.480, 103.681)),     # Lys57 NZ - G31 OP1
                          ((36.005, 14.643, 102.968), (38.180, 12.820, 102.681)),     # Arg63 NH1 - G31 OP2
                          ((37.154, 15.214, 101.097), (38.180, 12.820, 102.681)),     # Arg63 NH2 - G31 OP2
                          ((34.420, 9.880, 102.912), (32.988, 10.918, 105.251))],     # Arg33 NH2 - T30 OP2
                   title='estrogen receptor DNA-binding domain on DNA (1HCQ): Arg56 and Arg63 (green), Lys57 (orange) and Arg33 (magenta) of chain A '
                         'on the phosphates of T30 and G31 of chain D (cyan); orange dashed lines: N...O closer than 3.2 Å')
uridines = dict(pdb='1EHZ', highlights=[{'resn': 'U'}], title='tRNA (1EHZ): all the uridines. Find one stacked on another base, and one in a base pair')
CONTEXT = {
    'uracil_hbond': uridines, 'uracil_stack': uridines,
    'at_wc': dict(pdb='1BNA', highlights=[{'chain': 'A', 'resi': 6}, {'chain': 'B', 'resi': 19}], title='DNA (1BNA): the A-T pair A6-T19'),
    'gc_wc': dict(pdb='1BNA', highlights=[{'chain': 'A', 'resi': 3}, {'chain': 'B', 'resi': 22}], title='DNA (1BNA): the G-C pair C3-G22'),
    'indole_benzene': kinase([261, 264], 'Trp261 stacked on Tyr264'),
    'dms_benzene': kinase([458, 430], 'the sulfur of Met458 on the ring of Trp430'),
    'acetate_methylguanidinium': kinase([239, 308], 'the salt bridge between Arg239 and Glu308'),
    'methylammonium_indole': kinase([367, 405], 'Arg367 on the ring of Trp405 (a cation-pi contact; Lys can do the same)'),
    'phosphate_methylguanidinium': dna_binding, 'phosphate_methylammonium': dna_binding,
    'nma_dimer': dict(local_file=KINASE, highlights=[], lines=backbone_hbonds(KINASE),
                      title='Abl kinase (2GQG): the hydrogen bonds of the backbone (orange dashed lines) in helices and sheets'),
}
shown = []
for name in results:
    c = CONTEXT.get(name)
    if c is not None and c not in shown:
        shown.append(c)
        show_pdb_context(c.get('pdb', ''), c['highlights'], title=c['title'], labels=c.get('labels', ()),
                         lines=c.get('lines', ()), local_file=c.get('local_file'))

*Your text here.*

## 4. Discussion

*Interpret the results of the whole group. Some points to consider:*

*   *Which interactions are strong in vacuum, and which remain strong in water? What does this tell you about the forces that stabilise the double helix and the folded proteins, and about the role of the hydrophobic effect (which a continuum model of water does not describe)?*
*   *How reliable are your numbers? Where did the fast and the accurate methods agree, and where not?*
*   *A force field describes these interactions with fixed point charges and a Lennard-Jones term. From your results (the ESP, the role of dispersion, the decay with the distance), which interactions should it reproduce well, and which are more difficult (think of polarisation, and of cation–π)?*

## 5. Conclusions

## References

*   Luscombe NM, Laskowski RA, Thornton JM (2001) Amino acid–base interactions: a three-dimensional analysis of protein–DNA interactions at an atomic level. *Nucleic Acids Res* 29:2860
*   Jurečka P et al. (2006) Benchmark database of accurate (MP2 and CCSD(T) complete basis set limit) interaction energies of small model complexes, DNA base pairs, and amino acid pairs. *Phys Chem Chem Phys* 8:1985
*   Takatani T et al. (2010) Basis set consistent revision of the S22 test set of noncovalent interaction energies. *J Chem Phys* 132:144104
*   Grimme S et al. (2021) r$^2$SCAN-3c: a "Swiss army knife" composite electronic-structure method. *J Chem Phys* 154:064103
*   Bannwarth C, Ehlert S, Grimme S (2019) GFN2-xTB: an accurate and broadly parametrized self-consistent tight-binding quantum chemical method. *J Chem Theory Comput* 15:1652
*   *add the papers you used*

## 6. Going further (optional)

Two possible extensions, in different directions: choose one or both, or design your own. Add the results to the report and to the discussion.

### 6.1 More than two molecules: cooperativity of hydrogen bonds

Are interactions additive? In an α-helix or a β-sheet the backbone hydrogen bonds form chains (···H–N–C=O···H–N–C=O···), and each molecule of the chain is polarised by its neighbours: a hydrogen bond in a chain can be stronger than an isolated one. This *cooperativity* helps to explain why helices form all at once rather than one hydrogen bond at a time, and it is missing in force fields with fixed charges.

The cell builds chains of 2, 3 and 4 *N*-methylacetamide molecules, by repeating the arrangement of the dimer, and computes the total interaction energy (the energy of the chain minus that of the separated molecules) with the fast and the accurate methods. **Time**: about 10 minutes.

*Does the average energy per hydrogen bond change with the length of the chain? By how much (in kcal/mol and in %)? Does the fast method see the same effect? Look at the distance between the first and the third molecule: is the extra energy due to a direct interaction between them? What would you expect for a long helix?*

In [ ]:
%%group 1
xyzfile, nA, _, _ = complex_info('nma_dimer')
el, x = read_xyz(xyzfile)
A, B = x[:nA], x[nA:]
# the rotation and translation that bring molecule A onto molecule B: applied again and again, they build a chain
pa, pb = A.mean(axis=0), B.mean(axis=0)
U, S, Vt = np.linalg.svd((A - pa).T @ (B - pb))
R = Vt.T @ np.diag([1, 1, np.sign(np.linalg.det(Vt.T @ U.T))]) @ U.T
chain = [A, B]
while len(chain) < 4:
    chain.append((chain[-1] - pa) @ R.T + pb)
elA = el[:nA]

def energy(method, coords, tag):
    block = '\n'.join(f'{e} {p[0]:.5f} {p[1]:.5f} {p[2]:.5f}' for e, p in zip(elA * (len(coords) // nA), coords))
    keywords = f'! {method}' + ('' if 'XTB' in method else ' TightSCF')
    return final_energy(run_orca(tag, f'{keywords}\n* xyz 0 1\n{block}\n*', folder=os.path.join('g1', 'cooperativity', re.sub(r'[^A-Za-z0-9_+-]', '', method))))

rows = []
for method in [FAST, ACCURATE]:
    monomers = [energy(method, m, f'monomer_{k}') for k, m in enumerate(chain)]
    for n in [2, 3, 4]:
        e = (energy(method, np.vstack(chain[:n]), f'chain_{n}') - sum(monomers[:n])) * HARTREE2KCAL
        rows.append({'method': method, 'molecules': n, 'hydrogen bonds': n - 1, 'total (kcal/mol)': e, 'per hydrogen bond': e / (n - 1)})
coop = pd.DataFrame(rows)
display(coop.round(2))
with open(os.path.join('g1', 'cooperativity', 'nma_chain_4.xyz'), 'w') as f:
    f.write(f'{4 * nA}\nchain of 4 N-methylacetamide molecules\n' +
            '\n'.join(f'{e} {p[0]:.4f} {p[1]:.4f} {p[2]:.4f}' for e, p in zip(elA * 4, np.vstack(chain))) + '\n')
show_molecule(os.path.join('g1', 'cooperativity', 'nma_chain_4.xyz'))
print('distance between the centres of molecules 1 and 3:', round(float(np.linalg.norm(chain[2].mean(0) - chain[0].mean(0))), 1), 'Å')
fig, ax = plt.subplots(figsize=(5, 3.5))
for method, d in coop.groupby('method'):
    ax.plot(d['molecules'], d['per hydrogen bond'], 'o-', label=method)
ax.set_xlabel('molecules in the chain'); ax.set_ylabel('energy per hydrogen bond (kcal/mol)'); ax.set_xticks([2, 3, 4]); ax.legend()
plt.tight_layout(); save_figure(fig, 'g1_cooperativity'); plt.show()

*Your text here.*

### 6.2 At room temperature: do the complexes stay together?

Our energies are for a single, optimised structure, but at room temperature the molecules move. Here we run a short molecular dynamics (as in Part 5 of the training) of one of your complexes in water (described as a continuum) with the fast method, and follow the distance between the two molecules. **Time**: about 3–4 minutes per picosecond for the complexes with 20–30 atoms.

*Do the two molecules stay together? How much does the distance fluctuate? Does the arrangement change (e.g. a stacked pair sliding, a hydrogen bond breaking and forming again)? Try a higher temperature (e.g. 400 K): which complexes fall apart first, and is this consistent with their interaction energies in water? Why can't we see the complexes of the weakest interactions break in such a short simulation at 300 K? (Think of the time scale of a dissociation.)*

In [ ]:
%%group 1
MD_COMPLEX = my_complexes[0]
MD_PS = 3                    # length of the simulation (ps)
MD_TEMPERATURE = 300         # K

xyzfile, nA, qA, qB = complex_info(MD_COMPLEX)
md = f'''! Native-GFN2-XTB ALPB(water) MD
%md
  Timestep 1.0_fs
  Randomize 1
  Initvel {MD_TEMPERATURE}_K
  Thermostat Berendsen {MD_TEMPERATURE}_K Timecon 100_fs
  Dump Position Stride 20 Filename "trajectory.xyz"
  Run {1000 * MD_PS}
end
* xyz {qA + qB} 1
{xyz_block(xyzfile)}
*'''
name = f'md_{MD_TEMPERATURE}K_{MD_PS}ps'
run_orca(name, md, folder=os.path.join('g1', MD_COMPLEX))
traj = os.path.join('g1', MD_COMPLEX, name, 'trajectory.xyz')
lines = open(traj).read().splitlines()
n = int(lines[0])
frames = [np.array([[float(v) for v in l.split()[1:4]] for l in lines[i + 2:i + 2 + n]]) for i in range(0, len(lines) - n - 1, n + 2)]
el, _ = read_xyz(xyzfile)
heavy = np.array([e != 'H' for e in el])
t = np.arange(len(frames)) * 0.02                     # one frame every 20 fs
centres = [np.linalg.norm(f[nA:].mean(0) - f[:nA].mean(0)) for f in frames]
contact = [np.linalg.norm(f[:nA][heavy[:nA]][:, None] - f[nA:][heavy[nA:]][None], axis=2).min() for f in frames]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(t, centres, label='distance between the centres')
ax.plot(t, contact, label='closest heavy atoms')
ax.set_xlabel('time (ps)'); ax.set_ylabel('distance (Å)'); ax.legend(); ax.set_title(f'{MD_COMPLEX}, {MD_TEMPERATURE} K, water', loc='left')
plt.tight_layout(); save_figure(fig, f'g1_md_{MD_COMPLEX}_{MD_TEMPERATURE}K'); plt.show()

view = py3Dmol.view(width=500, height=350)
view.addModelsAsFrames(open(traj).read(), 'xyz')
view.setStyle({'stick': {'radius': 0.12}})
view.animate({'loop': 'forward', 'interval': 60})
view.zoomTo()
view.show()

*Your text here.*

# GROUP 2, Part 1: running the calculations

Write the names of your molecules and run the cells below. The file `settings.csv` gives, for each molecule, the bond that is twisted in lens 2 and the atom close to which a point charge is placed in lens 3.

In [ ]:
%%group 2
my_molecules = ['hbdi_neutral', 'hbdi_anion']

CHROMO_DIR = os.path.join(QM_DIR, 'chromophores')
settings = pd.read_csv(os.path.join(CHROMO_DIR, 'settings.csv')).set_index('molecule')
display(settings)
METHOD = 'CAM-B3LYP def2-SVP'

def charge_of(name):
    return int(re.search(r'charge=([+-]?\d+)', open(os.path.join(CHROMO_DIR, name + '.xyz')).readlines()[1]).group(1))

def n_electrons(xyzfile, charge):
    Z = {'H': 1, 'C': 6, 'N': 7, 'O': 8, 'S': 16}
    el, _ = read_xyz(xyzfile)
    return sum(Z[e] for e in el) - charge

def tddft(name, xyzfile, charge, folder, solvent=None, cubes=False, nroots=6, extra=''):
    # excited states with TD-DFT; returns the list of (energy eV, wavelength nm, oscillator strength)
    homo = n_electrons(xyzfile, charge) // 2 - 1
    keywords = f'! {METHOD} TightSCF' + (f' CPCM({solvent})' if solvent else '')
    plots = plots_block([f'MO("{name}.homo.cube",{homo},0)', f'MO("{name}.lumo.cube",{homo + 1},0)']) if cubes else ''
    inp = f'{keywords}\n{extra}%tddft\n  NRoots {nroots}\nend\n{plots}* xyz {charge} 1\n{xyz_block(xyzfile)}\n*'
    return excitations(run_orca(name, inp, folder=folder)).tolist()

def fmin(m):
    # oscillator strength above which a state is "bright": the bands of the indole are very weak
    return 0.001 if m == '3_methylindole' else 0.01

def lowest_bright(states, f_min=0.01):
    # the lowest excited state with oscillator strength above f_min: the first absorption band
    states = np.array(states)
    return states[np.argmax(states[:, 2] > f_min)]

def results_file(m):
    return os.path.join('g2', 'results', m + '.json')

def load_results(m):
    return json.load(open(results_file(m))) if os.path.isfile(results_file(m)) else {'molecule': m, 'charge': charge_of(m)}

def store(m, key, value):
    res = load_results(m)
    res[key] = value
    save_json(res, results_file(m))

for m in my_molecules:
    print(m, 'charge', charge_of(m))
    show_molecule(os.path.join(CHROMO_DIR, m + '.xyz'))

### Lens 1: the molecule at its minimum

The structure is optimised with GFN2-xTB, then TD-DFT gives the excited states (in vacuum) and writes the HOMO and the LUMO.

In [ ]:
%%group 2
def optimised(m):
    out = run_orca(m + '_opt', f'! Native-GFN2-XTB Opt\n* xyz {charge_of(m)} 1\n{xyz_block(os.path.join(CHROMO_DIR, m + ".xyz"))}\n*',
                   folder=os.path.join('g2', m))
    return os.path.join(os.path.dirname(out), m + '_opt.xyz')

for m in my_molecules:
    states = tddft(m + '_vacuum', optimised(m), charge_of(m), os.path.join('g2', m), cubes=True)
    store(m, 'vacuum', states)
    print(m)
    transitions(os.path.join('g2', m, m + '_vacuum', m + '_vacuum.out'))
    e, lam, f = lowest_bright(states, fmin(m))
    print(f'   first bright state: {e:.2f} eV, {lam:.0f} nm, f = {f:.3f}\n')

### Lens 2: the geometry

For the molecules with a flexible bond (`twist_atoms` in `settings.csv`), the bond is twisted by 30, 60 and 90 degrees from the planar structure: for each angle the rest of the molecule is optimised with GFN2-xTB, keeping the dihedral angle fixed, and the excited states are recomputed. Lumiflavin and the indole are rigid: for them we test another aspect of the geometry, recomputing the excited states at the structure optimised with the more accurate r$^2$SCAN-3c instead of GFN2-xTB (about 5 minutes for lumiflavin).

In [ ]:
%%group 2
def dihedral(x, i, j, k, l):
    b0, b1, b2 = x[i] - x[j], x[k] - x[j], x[l] - x[k]
    b1 = b1 / np.linalg.norm(b1)
    v, w = b0 - (b0 @ b1) * b1, b2 - (b2 @ b1) * b1
    return np.degrees(np.arctan2(np.cross(b1, v) @ w, v @ w))

for m in my_molecules:
    folder = os.path.join('g2', m)
    xyz, q = optimised(m), charge_of(m)
    twist = settings.loc[m, 'twist_atoms']
    if isinstance(twist, str):
        i, j, k, l = map(int, twist.split())
        el, x = read_xyz(xyz)
        phi0 = dihedral(x, i, j, k, l)            # close to 180 (trans) or to 0 (cis) at the minimum
        scan = {'angle': [0], 'states': [load_results(m)['vacuum']]}
        for angle in [30, 60, 90]:
            # the dihedral angle moves away from the planar value by "angle" degrees
            target = np.sign(phi0) * (180.0 - angle) if abs(phi0) > 90 else (1 if phi0 >= 0 else -1) * angle
            name = f'{m}_twist{angle}'
            out = run_orca(name, f'! Native-GFN2-XTB Opt\n%geom\n  Constraints\n    {{D {i} {j} {k} {l} {target:.1f} C}}\n  end\nend\n'
                                 f'* xyz {q} 1\n{xyz_block(xyz)}\n*', folder=folder)
            scan['angle'].append(angle)
            scan['states'].append(tddft(name + '_td', os.path.join(os.path.dirname(out), name + '.xyz'), q, folder, nroots=4))
        store(m, 'twist', scan)
        print(m, 'twisted around', settings.loc[m, 'twist_description'])
        for a, st in zip(scan['angle'], scan['states']):
            e, lam, f = lowest_bright(st, fmin(m))
            print(f'   {a:3d} degrees: {e:.2f} eV, {lam:.0f} nm, f = {f:.3f}')
    else:
        out = run_orca(m + '_r2scan', f'! r2SCAN-3c Opt TightSCF\n* xyz {q} 1\n{xyz_block(xyz)}\n*', folder=folder)
        states = tddft(m + '_r2scan_td', os.path.join(os.path.dirname(out), m + '_r2scan.xyz'), q, folder)
        store(m, 'r2scan_geometry', states)
        for label, st in [('GFN2-xTB structure', load_results(m)['vacuum']), ('r2SCAN-3c structure', states)]:
            e, lam, f = lowest_bright(st, fmin(m))
            print(f'{m}, {label}: {e:.2f} eV, {lam:.0f} nm, f = {f:.3f}')

### Lens 3: the environment

The excited states are recomputed (a) in water, as a continuum (CPCM), and (b) with a single point charge (an ion, or a charged residue of a protein) placed at 3 and 5 Å from the atom `charge_site` of `settings.csv`, on the outside of the molecule (along the line from the centre of the molecule to that atom). The sign of the charge is opposite to that of the charged group (a positive charge near a phenolate, a negative counterion near the Schiff base, as Glu113 in rhodopsin); for neutral molecules it is positive near an acceptor (O, N) and negative near an N–H. You can change it with `PC_CHARGE`.

In [ ]:
%%group 2
PC_CHARGE = {}          # e.g. {'hbdi_anion': -1.0} to change the default sign
PC_DISTANCES = [3.0, 5.0]

for m in my_molecules:
    folder = os.path.join('g2', m)
    xyz, q = optimised(m), charge_of(m)
    env = {'water': tddft(m + '_water', xyz, q, folder, solvent='water')}
    el, x = read_xyz(xyz)
    site = int(settings.loc[m, 'charge_site'])
    has_h = any(np.linalg.norm(x[h] - x[site]) < 1.15 for h in range(len(el)) if el[h] == 'H')
    qpc = PC_CHARGE.get(m, -1.0 if q > 0 or (q == 0 and has_h) else 1.0)
    direction = (x[site] - x.mean(axis=0)) / np.linalg.norm(x[site] - x.mean(axis=0))
    for d in PC_DISTANCES:
        p = x[site] + d * direction
        pcfile = os.path.abspath(os.path.join(folder, f'charge_{qpc:+.0f}_{d}.pc'))     # outside the calculation folder
        with open(pcfile, 'w') as f:
            f.write(f'1\n{qpc} {p[0]:.4f} {p[1]:.4f} {p[2]:.4f}\n')
        env[f'charge {qpc:+.0f} at {d} Å'] = tddft(f'{m}_pc_{qpc:+.0f}_{d}', xyz, q, folder, nroots=4, extra=f'%pointcharges "{pcfile}"\n')
    store(m, 'environment', env)
    print(m, f'(point charge {qpc:+.0f} near atom {site}, {el[site]})')
    for label, st in [('vacuum', load_results(m)['vacuum'])] + list(env.items()):
        e, lam, f = lowest_bright(st, fmin(m))
        print(f'   {label:22s} {e:.2f} eV, {lam:.0f} nm, f = {f:.3f}')

### Lens 4: the motion

A short molecular dynamics at 300 K with GFN2-xTB (in vacuum, as in Part 5 of the training), and TD-DFT on 6 snapshots, one every 100 fs after the first 300 fs. **Time**: about 3 minutes for the dynamics, plus 6 TD-DFT calculations (15–30 minutes in total).

In [ ]:
%%group 2
N_SNAPSHOTS = 6

for m in my_molecules:
    folder = os.path.join('g2', m, 'dynamics')
    xyz, q = optimised(m), charge_of(m)
    md = f'''! Native-GFN2-XTB MD
%md
  Timestep 0.5_fs
  Randomize 1
  Initvel 300_K
  Thermostat Berendsen 300_K Timecon 50_fs
  Dump Position Stride 200 Filename "trajectory.xyz"
  Run {200 * (N_SNAPSHOTS + 3)}
end
* xyz {q} 1
{xyz_block(xyz)}
*'''
    run_orca(m + '_md', md, folder=folder)
    lines = open(os.path.join(folder, m + '_md', 'trajectory.xyz')).read().splitlines()
    n = int(lines[0])
    frames = [lines[i:i + n + 2] for i in range(0, len(lines) - n - 1, n + 2)][3:3 + N_SNAPSHOTS]
    snapshots = []
    for k, frame in enumerate(frames):
        snap = os.path.join(folder, f'snapshot_{k}.xyz')
        with open(snap, 'w') as f:
            f.write('\n'.join(frame) + '\n')
        snapshots.append(tddft(f'{m}_snapshot_{k}', snap, q, folder, nroots=4))
    store(m, 'snapshots', snapshots)
    bright = np.array([lowest_bright(s, fmin(m)) for s in snapshots])
    print(f'{m}: minimum {lowest_bright(load_results(m)["vacuum"], fmin(m))[0]:.2f} eV, '
          f'snapshots {bright[:, 0].mean():.2f} ± {bright[:, 0].std():.2f} eV')

### GFP in its protein

The chromophore of GFP is formed by the protein itself, from three of its residues (Thr65, Tyr66 and Gly67 in the S65T mutant), and sits in the middle of a β-barrel. We use the structure of PDB 1EMA. The files in `Data/qm/gfp` were prepared as follows:

*   **the QM part**: the conjugated part of the chromophore, cut where it joins the protein backbone and capped with methyl groups. This is exactly HBDI. It was optimised with r$^2$SCAN-3c (in water, as a continuum) keeping the two twists of the bridge found in the crystal, and placed at the position of the chromophore in the crystal;
*   **the protein**: hydrogens added and protonation states chosen at pH 7 with `pdb2pqr` and `propka` (as in the docking practical), and every atom of the protein and of the 95 crystal waters given its point charge from the AMBER force field; the charges of the four backbone atoms bonded to the chromophore are moved to the neighbouring atoms.

There are two versions of the protein, one for each form of the chromophore: the extra proton is on Glu222 when the chromophore is anionic (as in S65T), on the chromophore when it is neutral, and His148 changes tautomer accordingly.

The first cell computes the two forms in vacuum (at the geometry they have in the protein) and in the field of the protein; the second runs the virtual mutations: the charges of the side chain of one residue (or of one water molecule) are switched off, which is a mutation to glycine as far as electrostatics is concerned. **Time**: about 5 minutes per calculation. The mutations can be split among several people, with `MY_MUTATIONS`; whoever runs only mutations sets `RUN_FORMS = False` in the first cell.

In [ ]:
%%group 2
GFP_DIR = os.path.join(QM_DIR, 'gfp')
GFP_STATES = ['anion', 'neutral']

def gfp_tddft(state, name, pcfile=None):
    xyz = os.path.join(GFP_DIR, f'hbdi_gfp_{state}.xyz')
    extra = f'%pointcharges "{os.path.abspath(pcfile)}"\n' if pcfile else ''
    return tddft(name, xyz, -1 if state == 'anion' else 0, os.path.join('g2', 'gfp', state), nroots=4, extra=extra)

def gfp_store(key, value):
    # one file per calculation, so that several people can run GFP calculations at the same time
    save_json({'key': key, 'states': value}, os.path.join('g2', 'results', 'gfp', key.replace(' ', '_') + '.json'))

def gfp_load():
    res = {}
    for f in sorted(glob.glob(os.path.join('g2', 'results', 'gfp', '*.json'))):
        d = json.load(open(f))
        if os.path.basename(f) == d.get('key', '').replace(' ', '_') + '.json':
            res[d['key']] = d['states']
    return res

RUN_FORMS = True     # False if someone else of the group computes the two forms (e.g. you only run some mutations)

if RUN_FORMS:
    for state in GFP_STATES:
        gfp_store(f'{state} vacuum', gfp_tddft(state, 'vacuum'))
        gfp_store(f'{state} protein', gfp_tddft(state, 'protein', os.path.join(GFP_DIR, f'gfp_{state}.pc')))
sync_from_drive()
gfp = gfp_load()
for key, st in gfp.items():
    if 'mutation' not in key:
        e, lam, f = lowest_bright(st)
        print(f'{key:18s} {e:.2f} eV, {lam:.0f} nm, f = {f:.2f}')

In [ ]:
%%group 2
MUTATIONS = {'Arg96': 96, 'His148': 148, 'Glu222': 222, 'Thr203': 203, 'Ser205': 205, 'Tyr145': 145, 'Gln94': 94, 'Gln69': 69, 'water 395': 395}
MY_MUTATIONS = list(MUTATIONS)        # or a part of them, e.g. ['Arg96', 'His148', 'Glu222']

BACKBONE = ('N', 'H', 'CA', 'HA', 'C', 'O')
for state in GFP_STATES:
    charges_table = pd.read_csv(os.path.join(GFP_DIR, f'gfp_{state}_charges.csv'))
    for label in MY_MUTATIONS:
        resid = MUTATIONS[label]
        q = charges_table.q.copy()
        off = (charges_table.resid == resid) & ~charges_table.atom.isin(BACKBONE)   # side chain (or the whole water molecule)
        q[off] = 0.0
        pcfile = os.path.join('g2', 'gfp', state, f'mutation_{resid}.pc')
        os.makedirs(os.path.dirname(pcfile), exist_ok=True)
        with open(pcfile, 'w') as f:
            f.write(f'{len(q)}\n' + ''.join(f'{qq:9.4f} {x:9.3f} {y:9.3f} {z:9.3f}\n'
                                             for qq, x, y, z in zip(q, charges_table.x, charges_table.y, charges_table.z)))
        states = gfp_tddft(state, f'mutation_{resid}', pcfile)
        gfp_store(f'{state} mutation {label}', states)
        ref = gfp_load().get(state + ' protein')
        print(f'{state:8s} without the side chain of {label:9s}: {lowest_bright(states)[1]:.0f} nm'
              + (f' (protein: {lowest_bright(ref)[1]:.0f} nm)' if ref else ''))

# GROUP 2, Part 2: the report

The cells of the report read the results of all the members of the group (run them again when the others have finished: `sync_from_drive()` brings their results here).

## 1. Introduction

*Explain to someone who has not studied this topic: why most biomolecules are colourless, what makes a molecule coloured, and why colour matters in biology (vision, light sensors, fluorescent proteins as tools in the lab). Which chromophores did your group study, in which proteins are they found, and what question did each member ask?*

*Your text here.*

## 2. Methods

*Describe in a few sentences, without equations, what you computed and how: how the structures were obtained; what TD-DFT computes and how a spectrum and a colour are obtained from it; how the geometry, the solvent, a point charge and the motion were included; how the protein was described, and what a virtual mutation is.*

*Your text here.*

## 3. Results

In [ ]:
%%group 2
sync_from_drive()
results = {}
for f in sorted(glob.glob(os.path.join('g2', 'results', '*.json'))):
    name = os.path.basename(f)[:-5]
    if name == 'gfp':
        continue
    res = json.load(open(f))
    if res.get('molecule') == name and 'vacuum' in res:
        results[name] = res
    else:              # e.g. a copy made by Google Drive ('psb_5 (1).json'), or a molecule without the first lens
        print('ignored:', f)
gfp = gfp_load()
print('molecules of the group:', list(results), '; GFP calculations:', len(gfp))

### 3.1 Which part of the molecule absorbs the light?

**Figure 1** (screenshot, one per molecule): HOMO (left) and LUMO (right). The two views are linked: rotating one rotates the other.

*Which atoms contribute to the HOMO and the LUMO? Is the excitation spread over the whole conjugated system? Does the excitation move charge from one part of the molecule to another (a* charge transfer*)? Compare your two molecules.*

In [ ]:
%%group 2
def show_homo_lumo(m, isovalue=0.03, width=760, height=300):
    folder = os.path.join('g2', m, m + '_vacuum')
    view = py3Dmol.view(viewergrid=(1, 2), width=width, height=height, linked=True)
    for col, orb in enumerate(['homo', 'lumo']):
        cube = os.path.join(folder, f'{m}_vacuum.{orb}.cube')
        view.addModel(cube_atoms_xyz(cube), 'xyz', viewer=(0, col))
        data = cube_for_display(cube)
        view.addVolumetricData(data, 'cube', {'isoval': isovalue, 'color': 'red', 'opacity': 0.8}, viewer=(0, col))
        view.addVolumetricData(data, 'cube', {'isoval': -isovalue, 'color': 'blue', 'opacity': 0.8}, viewer=(0, col))
    view.setStyle({'stick': {'radius': 0.12}})
    view.zoomTo()
    return view.show()

for m in results:
    print(f'{m}: HOMO (left) and LUMO (right)')
    show_homo_lumo(m)

*Your text here.*

### 3.2 Spectra, colours and experiment

**Figure 2**: the spectra (in water) of all the molecules of the group, with the visible region in colour, and the predicted colours. **Figure 3**: computed vs experimental wavelength of the first absorption band. Write the experimental values in `EXPERIMENT` (they are approximate: check them and cite the papers).

*Which molecules absorb in the visible, and which colour do the calculations predict? How large are the errors, and are they systematic? Is the ordering of the molecules right?*

In [ ]:
%%group 2
EXPERIMENT = {          # nm, first absorption band in solution (check the values and cite the papers in the report)
    'hbdi_neutral': 370, 'hbdi_anion': 425, 'psb_5': None, 'pyp_thioester': 400,
    'lumiflavin_ox': 445, '3_methylindole': 280,
}
emin, emax = 1.5, 7.0
fig, ax = plt.subplots(figsize=(9, 4.5))
for lam in np.arange(380, 750, 2):
    ax.axvspan(1239.84 / (lam + 1), 1239.84 / (lam - 1), color=plt.cm.nipy_spectral(0.05 + 0.85 * (lam - 380) / 370), alpha=0.18, lw=0)
for m, r in results.items():
    st = np.array(r['environment']['water'] if 'environment' in r else r['vacuum'])
    E, eps = spectrum(st, emin=emin, emax=emax)
    line, = ax.plot(E, eps, label=m)
    ax.vlines(st[:, 0], 0, st[:, 2] * 3, colors=line.get_color(), lw=1)
ax.set_xlim(emin, emax); ax.set_xlabel('excitation energy (eV)'); ax.set_ylabel('absorption (arb. units)'); ax.legend(fontsize=9)
secax = ax.secondary_xaxis('top', functions=(lambda e: 1239.84 / np.maximum(e, 1e-3), lambda l: 1239.84 / np.maximum(l, 1e-3)))
secax.set_xlabel('wavelength (nm)')
plt.tight_layout(); save_figure(fig, 'g2_spectra'); plt.show()

fig, ax = plt.subplots(figsize=(2.2 * len(results), 2))
for k, (m, r) in enumerate(results.items()):
    ax.add_patch(plt.Rectangle((k * 2.2, 0), 1.0, 1.0, color=colour_of(np.array(r['environment']['water'] if 'environment' in r else r['vacuum'])), ec='grey'))
    ax.text(k * 2.2 + 0.5, -0.1, m, ha='center', va='top', fontsize=9)
ax.set_xlim(-0.6, 2.2 * len(results)); ax.set_ylim(-0.6, 1.1); ax.axis('off')
save_figure(fig, 'g2_colours'); plt.show()

rows = []
for m, r in results.items():
    st = r['environment']['water'] if 'environment' in r else r['vacuum']
    rows.append({'molecule': m, 'computed (nm)': lowest_bright(st, fmin(m))[1], 'experiment (nm)': EXPERIMENT.get(m)})
comparison = pd.DataFrame(rows).set_index('molecule')
display(comparison)
known = comparison[comparison['experiment (nm)'].notna()]
if len(known):
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    ax.scatter(known['experiment (nm)'], known['computed (nm)'], s=50)
    for m, row in known.iterrows():
        ax.annotate(m, (row['experiment (nm)'], row['computed (nm)']), fontsize=8, xytext=(4, -10), textcoords='offset points')
    lo, hi = min(known.min()) - 30, max(known.max()) + 30
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1); ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_xlabel('experiment (nm)'); ax.set_ylabel('computed, in water (nm)')
    plt.tight_layout(); save_figure(fig, 'g2_experiment'); plt.show()

*Your text here.*

### 3.3 The geometry

**Figure 4**: wavelength and oscillator strength of the first bright state as the flexible bond is twisted (and, for the rigid molecules, the effect of the method used for the structure).

*How does twisting change the colour and the intensity? Explain it with the conjugation: what happens to the overlap of the p orbitals across the twisted bond? How much energy does it cost to twist the bond in the ground state (look at the energies of the constrained optimisations)? Proteins often twist their chromophores by 10–20 degrees: how large would the effect be?*

In [ ]:
%%group 2
twisted = {m: r for m, r in results.items() if 'twist' in r}
if twisted:
    fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
    for m, r in twisted.items():
        b = np.array([lowest_bright(st, fmin(m)) for st in r['twist']['states']])
        axs[0].plot(r['twist']['angle'], b[:, 1], 'o-', label=m)
        axs[1].plot(r['twist']['angle'], b[:, 2], 'o-', label=m)
    axs[0].set_ylabel('wavelength of the first bright state (nm)'); axs[1].set_ylabel('oscillator strength')
    for a in axs:
        a.set_xlabel('twist of the bond (degrees)'); a.set_xticks([0, 30, 60, 90])
    axs[0].legend(fontsize=8)
    plt.tight_layout(); save_figure(fig, 'g2_twist'); plt.show()
for m, r in results.items():
    if 'r2scan_geometry' in r:
        print(f"{m}: {lowest_bright(r['vacuum'], fmin(m))[1]:.0f} nm at the GFN2-xTB structure, "
              f"{lowest_bright(r['r2scan_geometry'], fmin(m))[1]:.0f} nm at the r2SCAN-3c structure")

*Your text here.*

### 3.4 The environment

**Figure 5**: the shift of the first bright state (in eV, positive = higher energy = shorter wavelength) in water and with a point charge at 3 and 5 Å.

*Which molecules are most sensitive to the environment, and why (charged vs neutral, charge transfer)? Does the point charge shift the absorption to shorter or longer wavelengths? Explain the sign of the shift by thinking about where the charge of the molecule moves in the excited state (Figure 1): is the excited state stabilised or destabilised by the point charge, compared with the ground state?*

In [ ]:
%%group 2
rows = []
for m, r in results.items():
    if 'environment' in r:
        e0 = lowest_bright(r['vacuum'], fmin(m))[0]
        for label, st in r['environment'].items():
            rows.append({'molecule': m, 'environment': label.replace('+1', '±1').replace('-1', '±1'), 'shift (eV)': lowest_bright(st, fmin(m))[0] - e0})
if rows:
    env = pd.DataFrame(rows)
    table = env.pivot(index='molecule', columns='environment', values='shift (eV)')
    display(table.round(3))
    fig, ax = plt.subplots(figsize=(1.4 * len(table) + 3, 4))
    table.plot.bar(ax=ax, rot=20)
    ax.axhline(0, color='grey', lw=0.8); ax.set_ylabel('shift of the first bright state (eV)'); ax.set_xlabel('')
    plt.tight_layout(); save_figure(fig, 'g2_environment'); plt.show()

*Your text here.*

### 3.5 The motion

**Figure 6**: the spectra computed from the snapshots of the dynamics (each with a narrow band), compared with the spectrum of the minimum drawn with the usual width of 0.3 eV.

*How large is the spread of the excitation energies? Is the average different from the value at the minimum? Compare the width of the band obtained from the snapshots with the width of the experimental band (look at an experimental spectrum of your molecules). What is missing (more snapshots, the quantum nature of the nuclei, the solvent)?*

In [ ]:
%%group 2
with_md = {m: r for m, r in results.items() if 'snapshots' in r}
if with_md:
    fig, axs = plt.subplots(1, len(with_md), figsize=(4.2 * len(with_md), 3.5), squeeze=False)
    for ax, (m, r) in zip(axs[0], with_md.items()):
        e0 = lowest_bright(r['vacuum'], fmin(m))[0]
        emin, emax = e0 - 1.5, e0 + 1.5
        E = np.linspace(emin, emax, 1000)
        average = sum(spectrum(np.array(s), fwhm=0.1, emin=emin, emax=emax)[1] for s in r['snapshots']) / len(r['snapshots'])
        ax.plot(E, average, label='snapshots at 300 K')
        ax.plot(*spectrum(np.array(r['vacuum']), fwhm=0.3, emin=emin, emax=emax), '--', label='minimum, width 0.3 eV')
        ax.set_title(m, fontsize=10); ax.set_xlabel('excitation energy (eV)')
    axs[0, 0].set_ylabel('absorption (arb. units)'); axs[0, 0].legend(fontsize=8)
    plt.tight_layout(); save_figure(fig, 'g2_motion'); plt.show()

*Your text here.*

### 3.6 GFP in its protein

**Figure 7**: the absorption of the two forms of the GFP chromophore in vacuum and in the protein, with the experimental values. **Figure 8**: the effect of the virtual mutations (the shift of the first bright state when the side chain is switched off). **Figure 9** (screenshot): the protein around the chromophore, with the residues coloured by the shift they cause (red: removing the residue shifts the absorption to longer wavelengths; blue: to shorter).

*Does the protein shift the absorption of the chromophore? In which direction, and for which form? Which residues tune the colour most? Look at the structure: are they charged, do they make hydrogen bonds with the chromophore? Where are they relative to the phenolate and the imidazolinone ends? The isolated anion has also been measured in vacuum (in an ion storage ring): it absorbs at 479 nm, almost the same as in the protein (Nielsen et al., *Phys. Rev. Lett.* 87, 228102, 2001). Compare this with your calculation: what is larger, the effect of the protein or the error of the method? Remember that a mutation also changes the structure, which we don't include: how could you design a GFP variant with a different colour? (Look up the mutations of the blue, cyan and yellow fluorescent proteins.)*

In [ ]:
%%group 2
if gfp:
    exp_nm = {'anion': 489, 'neutral': 395}
    fig, ax = plt.subplots(figsize=(6, 3.5))
    for k, state in enumerate(GFP_STATES):
        for j, cond in enumerate(['vacuum', 'protein']):
            if f'{state} {cond}' in gfp:
                ax.bar(k * 3 + j, lowest_bright(gfp[f'{state} {cond}'])[1], color=['C0', 'C2'][j], label=cond if k == 0 else None)
        ax.scatter(k * 3 + 0.5, exp_nm[state], marker='_', s=1500, color='black', zorder=3, label='experiment' if k == 0 else None)
    ax.set_xticks([0.5, 3.5]); ax.set_xticklabels(['anion (S65T: 489 nm)', 'neutral (wild type: 395 nm)'])
    ax.set_ylabel('wavelength of the first bright state (nm)'); ax.legend(fontsize=8, loc='upper center', ncol=3); ax.set_ylim(250, 560)
    plt.tight_layout(); save_figure(fig, 'g2_gfp'); plt.show()

    rows = []
    for key, st in gfp.items():
        if 'mutation' in key:
            state, label = key.split(' mutation ')
            rows.append({'residue': label, 'state': state,
                         'shift (eV)': lowest_bright(st)[0] - lowest_bright(gfp[f'{state} protein'])[0]})
    if rows:
        mut = pd.DataFrame(rows).pivot(index='residue', columns='state', values='shift (eV)')
        display(mut.round(3))
        fig, ax = plt.subplots(figsize=(7, 3.8))
        mut.plot.bar(ax=ax, rot=30)
        ax.axhline(0, color='grey', lw=0.8); ax.set_ylabel('shift when the side chain is switched off (eV)'); ax.set_xlabel('')
        plt.tight_layout(); save_figure(fig, 'g2_gfp_mutations'); plt.show()

        state = 'anion'
        view = py3Dmol.view(width=700, height=450)
        view.addModel(open(os.path.join(GFP_DIR, 'gfp_protein.pdb')).read(), 'pdb')
        view.setStyle({'cartoon': {'color': 'lightgrey', 'opacity': 0.6}})
        view.addModel(open(os.path.join(GFP_DIR, f'hbdi_gfp_{state}.xyz')).read(), 'xyz')
        view.setStyle({'model': 1}, {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.2}})
        vmax = max(0.05, mut[state].abs().max())
        for label, shift in mut[state].items():
            resid = MUTATIONS[label]
            colour = plt.cm.RdBu_r(0.5 - 0.5 * shift / vmax)       # shift < 0 (red shift when removed): red
            hexcol = '#%02x%02x%02x' % tuple(int(255 * c) for c in colour[:3])
            view.addStyle({'model': 0, 'resi': resid}, {'stick': {'color': hexcol, 'radius': 0.18}})
            view.addResLabels({'model': 0, 'resi': resid}, {'fontSize': 10, 'showBackground': True, 'backgroundOpacity': 0.5})
        view.zoomTo({'model': 1}); view.zoom(0.6)
        view.show()

*Your text here.*

### 3.7 The other chromophores in their proteins

The cells below show the other chromophores in their proteins (downloaded by your browser from the PDB; if they don't appear, download them from [rcsb.org](https://www.rcsb.org) and open them in VMD or ChimeraX): rhodopsin (1U19, retinal bound to Lys296, with its counterion Glu113), the photoactive yellow protein (1NWZ, the chromophore bound to Cys69, with Tyr42 and Glu46 hydrogen-bonded to the phenolate), a DNA photolyase (1DNP, with its flavin FAD), and the Trp, Tyr and Phe residues of the Abl kinase (2GQG, local file).

**Figure 10** (screenshot): your chromophores in their proteins.

*Look at the environment of the chromophore: which groups are close to it (charges, hydrogen bonds, aromatic rings)? Is it planar or twisted? Using your results on geometry and environment, explain how the protein could tune its colour.*

In [ ]:
%%group 2
KINASE = os.path.join(REPO_DIR, 'Data', 'docking', '2gqg.pdb')
CONTEXT = {
    'psb_5': [dict(pdb='1U19', highlights=[{'resn': 'RET'}, {'chain': 'A', 'resi': [296, 113]}], labels=[{'chain': 'A', 'resi': [296, 113]}],
                   title='rhodopsin (1U19): 11-cis retinal, Lys296 and Glu113')],
    'pyp_thioester': [dict(pdb='1NWZ', highlights=[{'resn': 'HC4'}, {'resi': [42, 46, 69]}], labels=[{'resi': [42, 46, 69]}],
                           title='photoactive yellow protein (1NWZ): the chromophore, Cys69, Tyr42 and Glu46')],
    'lumiflavin_ox': [dict(pdb='1DNP', highlights=[{'resn': 'FAD'}], title='DNA photolyase (1DNP): the flavin (FAD)')],
    '3_methylindole': [dict(local_file=KINASE, highlights=[{'resn': 'TRP'}, {'resn': 'TYR'}, {'resn': 'PHE'}],
                            title='Abl kinase (2GQG): Trp (green), Tyr (orange) and Phe (cyan)')],
}
for m in results:
    for c in CONTEXT.get(m, []):
        show_pdb_context(c.get('pdb', ''), c['highlights'], title=c['title'], labels=c.get('labels', ()), local_file=c.get('local_file'))

*Your text here.*

## 4. Discussion

*Interpret the results of the whole group. Some points to consider:*

*   *Rank the four factors (chemistry, geometry, environment, motion) by how much they change the colour of your chromophores. Which ones can a protein control?*
*   *How reliable are the calculations, for the absolute colour and for the changes? What would you need to improve them?*
*   *Our model of the protein is a set of fixed point charges at the crystal structure. What is missing (the response of the protein to the excitation, the motion of the protein, the change of structure caused by a real mutation)? How would you include it? (Think about QM/MM and MD, lectures 5 and 8.)*

## 5. Conclusions

## References

*   Ormö M et al. (1996) Crystal structure of the *Aequorea victoria* green fluorescent protein. *Science* 273:1392 (PDB 1EMA)
*   Tsien RY (1998) The green fluorescent protein. *Annu Rev Biochem* 67:509
*   Nielsen SB et al. (2001) Absorption spectrum of the green fluorescent protein chromophore anion in vacuo. *Phys Rev Lett* 87:228102
*   Dreuw A, Head-Gordon M (2005) Single-reference ab initio methods for the calculation of excited states of large molecules. *Chem Rev* 105:4009
*   Bannwarth C, Ehlert S, Grimme S (2019) GFN2-xTB. *J Chem Theory Comput* 15:1652
*   *add the experimental papers you used (one for each experimental value)*

## 6. Going further (optional)

Two possible extensions, in different directions: choose one or both, or design your own. Add the results to the report and to the discussion.

### 6.1 Design your own GFP variant

With the same tools as the virtual mutations you can test ideas for new colours: switch off several residues at once, change a protonation state (e.g. make Glu222 charged while the chromophore is anionic), or put a new charge where a residue is (e.g. a +1 charge on the OG1 atom of Thr203, a crude model of a mutation to Lys). Each calculation takes about 5 minutes.

*Which changes give the largest shift towards the red and towards the blue? Are the effects of two mutations additive? Compare with real GFP variants (e.g. T203Y in the yellow fluorescent protein, which places an aromatic ring on top of the phenolate; Y66H in the blue fluorescent protein, which changes the chromophore itself). What can a point-charge model not describe?*

In [ ]:
%%group 2
VARIANT_NAME = 'T203K-like'
VARIANT_STATE = 'anion'
OFF = [203]                                   # residues whose side chain is switched off
NEW_CHARGES = [(203, 'OG1', +1.0)]            # (residue, atom, charge) placed at the position of an atom

table = pd.read_csv(os.path.join(GFP_DIR, f'gfp_{VARIANT_STATE}_charges.csv'))
q = table.q.copy()
q[table.resid.isin(OFF) & ~table.atom.isin(BACKBONE)] = 0.0
for resid, atom, charge in NEW_CHARGES:
    q[(table.resid == resid) & (table.atom == atom)] = charge
pcfile = os.path.join('g2', 'gfp', VARIANT_STATE, f'variant_{VARIANT_NAME}.pc')
with open(pcfile, 'w') as f:
    f.write(f'{len(q)}\n' + ''.join(f'{qq:9.4f} {x:9.3f} {y:9.3f} {z:9.3f}\n' for qq, x, y, z in zip(q, table.x, table.y, table.z)))
states = gfp_tddft(VARIANT_STATE, f'variant_{VARIANT_NAME}', pcfile)
gfp_store(f'{VARIANT_STATE} variant {VARIANT_NAME}', states)
ref = lowest_bright(gfp_load()[f'{VARIANT_STATE} protein'])
new = lowest_bright(states)
print(f'{VARIANT_NAME}: {new[1]:.0f} nm (wild type in our model: {ref[1]:.0f} nm), shift {new[0] - ref[0]:+.2f} eV')

*Your text here.*

### 6.2 Fluorescence: the Stokes shift

After absorbing light, a molecule relaxes in the excited state: its structure changes, and the light it emits (fluorescence) has a longer wavelength than the light it absorbed. The difference is the **Stokes shift**, which is what makes fluorescence microscopy possible (the emitted light can be separated from the excitation light). Here we optimise the structure of the first excited state with TD-DFT and compute the emission energy, the energy difference between the excited and the ground state at that structure. **Time**: about 40 minutes for 3-methylindole (Trp fluorescence, ≈ 350 nm; each step of the optimisation needs a full TD-DFT calculation and its gradient), 2–3 hours for lumiflavin (≈ 530 nm). The optimisation starts from the GFN2-xTB structure, so a small part of the shift you compute (≈ 0.05 eV, compare the two structures in Lens 2) comes from the change of method, not from the excited state. The chains with a flexible double bond (`psb_5`, `pyp_thioester`) twist in the excited state, which is how they work as photoswitches, but which TD-DFT cannot follow to the end.

*How large is the Stokes shift, in nm and in eV? Which bonds change length in the excited state (compare the two structures), and how is this related to the HOMO and LUMO? Why is the fluorescence of Trp used to follow protein folding?*

In [ ]:
%%group 2
EMISSION_MOLECULE = '3_methylindole'
EMISSION_ROOT = 1            # the excited state that emits: the lowest one (for the indole, look at which state is bright)

m = EMISSION_MOLECULE
folder = os.path.join('g2', m, 'emission')
xyz, q = optimised(m), charge_of(m)
out = run_orca(m + '_S1_opt', f'! {METHOD} Opt\n%tddft\n  NRoots 3\n  IRoot {EMISSION_ROOT}\nend\n* xyz {q} 1\n{xyz_block(xyz)}\n*', folder=folder)
emission = excitations(out)[EMISSION_ROOT - 1]
absorption = np.array(load_results(m)['vacuum'])[EMISSION_ROOT - 1]
store(m, 'emission', {'absorption': absorption.tolist(), 'emission': emission.tolist()})
print(f'{m}: absorption {absorption[0]:.2f} eV ({absorption[1]:.0f} nm), emission {emission[0]:.2f} eV ({emission[1]:.0f} nm), '
      f'Stokes shift {absorption[0] - emission[0]:.2f} eV ({emission[1] - absorption[1]:.0f} nm)')
el, x0 = read_xyz(xyz)
_, x1 = read_xyz(os.path.join(os.path.dirname(out), m + '_S1_opt.xyz'))
heavy = [i for i, e in enumerate(el) if e != 'H']
print('bonds that change by more than 0.02 Å in the excited state:')
for a in heavy:
    for b in heavy:
        if a < b and np.linalg.norm(x0[a] - x0[b]) < 1.6:
            d0, d1 = np.linalg.norm(x0[a] - x0[b]), np.linalg.norm(x1[a] - x1[b])
            if abs(d1 - d0) > 0.02:
                print(f'   {el[a]}{a}-{el[b]}{b}: {d0:.3f} -> {d1:.3f} Å')

*Your text here.*